# Antidepressant Drug Class Cleaning, Index-Date Distribution, and LOT Analysis

This notebook prepares the antidepressant drug exposure data for Tables 4–7.

Major sections:

1. Antidepressant generic-name and class mapping
2. Index-date antidepressant class distribution
3. Mono-class cohort checks and duration analysis
4. Table 6 LOT1 to LOT2 transition summaries and Sankey plots
5. Table 7 single-class LOT2 overlap summaries and SSRI-focused overlap distribution

Upload note: outputs have been cleared so the notebook can be stored without displaying patient-level records in GitHub.


## Import antidepressant drug exposure records

This query imports antidepressant drug exposure records for the MDD + antidepressant cohort.


In [ ]:
import pandas
import os

# This query represents dataset "tb4_Drug_MDD + Antidepressants (N=68301)" for domain "drug" and was generated for All of Us Controlled Tier Dataset v8
dataset_80206130_drug_sql = """
    SELECT
        d_exposure.person_id,
        d_exposure.drug_concept_id,
        d_standard_concept.concept_name as standard_concept_name,
        d_standard_concept.concept_code as standard_concept_code,
        d_standard_concept.vocabulary_id as standard_vocabulary,
        d_exposure.drug_exposure_start_datetime,
        d_exposure.drug_exposure_end_datetime,
        d_exposure.verbatim_end_date,
        d_exposure.drug_type_concept_id,
        d_type.concept_name as drug_type_concept_name,
        d_exposure.stop_reason,
        d_exposure.refills,
        d_exposure.quantity,
        d_exposure.days_supply,
        d_exposure.sig,
        d_exposure.route_concept_id,
        d_route.concept_name as route_concept_name,
        d_exposure.lot_number,
        d_exposure.visit_occurrence_id,
        d_visit.concept_name as visit_occurrence_concept_name,
        d_exposure.drug_source_value,
        d_exposure.drug_source_concept_id,
        d_source_concept.concept_name as source_concept_name,
        d_source_concept.concept_code as source_concept_code,
        d_source_concept.vocabulary_id as source_vocabulary,
        d_exposure.route_source_value,
        d_exposure.dose_unit_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.drug_exposure` d_exposure 
        WHERE
            (
                drug_concept_id IN (SELECT
                    DISTINCT ca.descendant_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                JOIN
                    (SELECT
                        DISTINCT c.concept_id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id             
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                        WHERE
                            concept_id IN (1366610, 1510996, 35603277, 37498659, 40234834, 43560354, 44507700, 46275300, 703470, 703547, 705755, 710062, 713109, 714684, 715259, 715939, 716968, 717607, 721724, 722031, 725131, 733896, 738156, 739138, 743670, 750982, 751412, 754270, 755695, 766209, 778268, 781705, 797617, 798834)             
                            AND full_text LIKE '%_rank1]%'       ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) b 
                        ON (ca.ancestor_id = b.concept_id)))  
                    AND (d_exposure.PERSON_ID IN (SELECT
                        distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                WHERE
                                    concept_id IN (4152280)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 )) criteria ) 
                    AND cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (798834, 35603277, 715939, 766209, 713109, 46275300, 721724, 781705, 755695, 754270, 738156, 715259, 739138, 1510996, 722031, 37498659, 797617, 778268, 43560354, 743670, 750982, 733896, 751412, 710062, 703547, 705755, 725131, 40234834, 703470, 716968, 44507700, 1366610, 714684, 717607)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria ) ))) d_exposure 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_standard_concept 
                ON d_exposure.drug_concept_id = d_standard_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_type 
                ON d_exposure.drug_type_concept_id = d_type.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_route 
                ON d_exposure.route_concept_id = d_route.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                ON d_exposure.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_visit 
                ON v.visit_concept_id = d_visit.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_source_concept 
                ON d_exposure.drug_source_concept_id = d_source_concept.concept_id"""

dataset_80206130_drug_df = pandas.read_gbq(
    dataset_80206130_drug_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_80206130_drug_df.head(5)

## Save raw working dataset

This step saves the imported drug exposure dataset for local notebook use inside the research workspace.


In [ ]:
# ============================================================
# SAVE RAW DATASET: 68,301 MDD + antidepressant patients
# ============================================================

import os

output_dir = "/home/jupyter/workspaces/mddprojectcontrolledtier/raw_data_backup"
os.makedirs(output_dir, exist_ok=True)

# 68,301-person raw drug records from Workbench query
raw_68301 = dataset_80206130_drug_df.copy()

print("68,301 dataset patients:", raw_68301["person_id"].nunique())
print("68,301 dataset records:", len(raw_68301))

# Save as Parquet
raw_68301_parquet_path = (
    output_dir
    + "/raw_drug_records_MDD_antidepressants_N68301.parquet"
)

raw_68301.to_parquet(
    raw_68301_parquet_path,
    index=False
)

# Save as CSV
raw_68301_csv_path = (
    output_dir
    + "/raw_drug_records_MDD_antidepressants_N68301.csv"
)

raw_68301.to_csv(
    raw_68301_csv_path,
    index=False
)

print("Saved 68,301 raw data:")
print(raw_68301_parquet_path)
print(raw_68301_csv_path)

In [ ]:
ad_drug_df = dataset_80206130_drug_df.copy()

print("Total antidepressant drug exposure records:", len(ad_drug_df))
print("Total unique patients:", ad_drug_df["person_id"].nunique())

ad_drug_df.head()

## Medication concept checks

These checks review bupropion and dextromethorphan-related records before final antidepressant mapping.


In [ ]:
# Look at all unique standard concept names containing bupropion or dextromethorphan
ad_drug_df[
    ad_drug_df["standard_concept_name"]
    .str.lower()
    .str.contains("bupropion|dextromethorphan", na=False)
][["drug_concept_id", "standard_concept_name", "standard_concept_code", "standard_vocabulary"]].drop_duplicates().sort_values(
    "standard_concept_name"
).head(100)

In [ ]:
ad_drug_df[
    ad_drug_df["standard_concept_name"]
    .str.lower()
    .str.contains("bupropion", na=False)
    &
    ad_drug_df["standard_concept_name"]
    .str.lower()
    .str.contains("dextromethorphan", na=False)
][["drug_concept_id", "standard_concept_name", "standard_concept_code", "standard_vocabulary"]].drop_duplicates().sort_values(
    "standard_concept_name"
)


## Antidepressant generic-name and class mapping

This section maps drug concept names to antidepressant generic names and therapeutic classes.


In [ ]:
import pandas as pd
import numpy as np

# Combination product concept IDs observed in your dataset
DEX_BUP_CONCEPT_IDS = {1525612, 1525618}

# Generic name to class mapping
ad_generic_to_class = {
    # SSRI
    "citalopram": "SSRI",
    "escitalopram": "SSRI",
    "fluoxetine": "SSRI",
    "fluvoxamine": "SSRI",
    "paroxetine": "SSRI",
    "sertraline": "SSRI",
    
    # SNRI
    "venlafaxine": "SNRI",
    "desvenlafaxine": "SNRI",
    "duloxetine": "SNRI",
    "levomilnacipran": "SNRI",
    
    # Atypical Antidepressants
    "bupropion": "Atypical Antidepressants",
    "vortioxetine": "Atypical Antidepressants",
    "vilazodone": "Atypical Antidepressants",
    "trazodone": "Atypical Antidepressants",
    "nefazodone": "Atypical Antidepressants",
    "mirtazapine": "Atypical Antidepressants",
    "esketamine": "Atypical Antidepressants",
    "brexanolone": "Atypical Antidepressants",
    "dextromethorphan + bupropion": "Atypical Antidepressants",
    
    # TCA
    "amitriptyline": "TCA",
    "nortriptyline": "TCA",
    "imipramine": "TCA",
    "desipramine": "TCA",
    "clomipramine": "TCA",
    "doxepin": "TCA",
    "trimipramine": "TCA",
    "protriptyline": "TCA",
    "amoxapine": "TCA",
    
    # MAOI
    "phenelzine": "MAOI",
    "tranylcypromine": "MAOI",
    "isocarboxazid": "MAOI",
    "selegiline": "MAOI",
}

# Match longer names first
ad_generic_order = sorted(ad_generic_to_class.keys(), key=len, reverse=True)

In [ ]:
# Start with empty generic name
ad_drug_df["ad_generic_name"] = np.nan

# Convert standard_concept_name to lowercase string
ad_drug_df["standard_name_lower"] = (
    ad_drug_df["standard_concept_name"]
    .astype(str)
    .str.lower()
)

# 1. First handle the combination drug by concept ID
ad_drug_df.loc[
    ad_drug_df["drug_concept_id"].isin(DEX_BUP_CONCEPT_IDS),
    "ad_generic_name"
] = "dextromethorphan + bupropion"

# 2. Backup rule: handle combination drug by name
ad_drug_df.loc[
    ad_drug_df["ad_generic_name"].isna()
    & ad_drug_df["standard_name_lower"].str.contains("dextromethorphan", na=False)
    & ad_drug_df["standard_name_lower"].str.contains("bupropion", na=False),
    "ad_generic_name"
] = "dextromethorphan + bupropion"

# 3. Match all other generic names
for generic in ad_generic_order:
    if generic == "dextromethorphan + bupropion":
        continue
    
    mask = (
        ad_drug_df["ad_generic_name"].isna()
        & ad_drug_df["standard_name_lower"].str.contains(generic, na=False)
    )
    
    ad_drug_df.loc[mask, "ad_generic_name"] = generic

# 4. Assign class
ad_drug_df["ad_class"] = ad_drug_df["ad_generic_name"].map(ad_generic_to_class)

print("Total records:", len(ad_drug_df))
print("Mapped records:", ad_drug_df["ad_generic_name"].notna().sum())
print("Unmapped records:", ad_drug_df["ad_generic_name"].isna().sum())

In [ ]:
# Add adjunct antipsychotic drugs into Atypical Antidepressants
# This is needed because these drugs were included in the original medication concept set,
# but Table 4 only summarizes antidepressant index class as:
# Atypical, MAOI, SNRI, SSRI, TCA, and Multiple index classes.

#Brexpiprazole, cariprazine, and lumateperone were included in the original medication concept
#set as adjunct medications. Since Table 4 summarizes medication exposure using the predefined
# antidepressant class categories and does not include a separate adjunct antipsychotic category,
#these adjunct medications were grouped under Atypical Antidepressants for the index-class summary.

adjunct_drugs = ["brexpiprazole", "cariprazine", "lumateperone"]

for drug in adjunct_drugs:
    mask = (
        ad_drug_df["ad_generic_name"].isna()
        & ad_drug_df["standard_name_lower"].str.contains(drug, na=False)
    )
    
    ad_drug_df.loc[mask, "ad_generic_name"] = drug
    ad_drug_df.loc[mask, "ad_class"] = "Atypical Antidepressants"

print("Total records:", len(ad_drug_df))
print("Mapped records:", ad_drug_df["ad_generic_name"].notna().sum())
print("Unmapped records:", ad_drug_df["ad_generic_name"].isna().sum())

In [ ]:
unmapped_ad = (
    ad_drug_df[ad_drug_df["ad_generic_name"].isna()]
    [["drug_concept_id", "standard_concept_name", "standard_concept_code", "standard_vocabulary"]]
    .drop_duplicates()
    .sort_values("standard_concept_name")
)

print("Unmapped unique standard concept names:", len(unmapped_ad))
unmapped_ad.head(100)

## Table 4: Index-date antidepressant class distribution

This section identifies each patient’s first observed antidepressant date and summarizes antidepressant class combinations at the index date.


In [ ]:
ad_class_record_check = (
    ad_drug_df
    .dropna(subset=["ad_class"])
    .groupby("ad_class", as_index=False)
    .agg(
        total_records=("person_id", "size"),
        unique_patients=("person_id", "nunique")
    )
    .sort_values("unique_patients", ascending=False)
    .reset_index(drop=True)
)

ad_class_record_check

In [ ]:
# Convert start datetime to date only
ad_drug_df["drug_exposure_start_datetime"] = pd.to_datetime(
    ad_drug_df["drug_exposure_start_datetime"],
    errors="coerce",
    utc=True
)

ad_drug_df["drug_exposure_start_date"] = (
    ad_drug_df["drug_exposure_start_datetime"].dt.date
)

# Keep only mapped antidepressant / adjunct records
ad_clean = ad_drug_df.dropna(
    subset=["ad_generic_name", "ad_class", "drug_exposure_start_date"]
).copy()

# Find first medication date per person
index_drug_date_df = (
    ad_clean
    .groupby("person_id", as_index=False)["drug_exposure_start_date"]
    .min()
    .rename(columns={"drug_exposure_start_date": "index_drug_date"})
)

# Keep records on index drug date
index_records = ad_clean.merge(
    index_drug_date_df,
    on="person_id",
    how="inner"
)

index_records = index_records[
    index_records["drug_exposure_start_date"] == index_records["index_drug_date"]
].copy()

# Patient-level class summary at index date
patient_index_class = (
    index_records
    .groupby("person_id")
    .agg(
        index_drug_date=("index_drug_date", "first"),
        n_class=("ad_class", lambda x: len(set(x))),
        class_combo=("ad_class", lambda x: " + ".join(sorted(set(x)))),
        n_generic=("ad_generic_name", lambda x: len(set(x))),
        generic_combo=("ad_generic_name", lambda x: " + ".join(sorted(set(x))))
    )
    .reset_index()
)

print("Total patients at index date:", patient_index_class["person_id"].nunique())
print("Mono-class patients:", (patient_index_class["n_class"] == 1).sum())
print("Multiple-class patients:", (patient_index_class["n_class"] > 1).sum())

In [ ]:
# Table 4: Index date drug class distribution

single_class_counts = (
    patient_index_class[patient_index_class["n_class"] == 1]
    .groupby("class_combo")["person_id"]
    .nunique()
)

multiple_class_n = patient_index_class.loc[
    patient_index_class["n_class"] > 1,
    "person_id"
].nunique()

tbl4_index_class_distribution = pd.DataFrame({
    "Index Drug Class": [
        "Atypical Antidepressants",
        "MAOI",
        "Multiple index classes",
        "SNRI",
        "SSRI",
        "TCA",
        "Total"
    ],
    "Patient N": [
        int(single_class_counts.get("Atypical Antidepressants", 0)),
        int(single_class_counts.get("MAOI", 0)),
        int(multiple_class_n),
        int(single_class_counts.get("SNRI", 0)),
        int(single_class_counts.get("SSRI", 0)),
        int(single_class_counts.get("TCA", 0)),
        int(patient_index_class["person_id"].nunique())
    ]
})

tbl4_index_class_distribution["% among 68301"] = (
    tbl4_index_class_distribution["Patient N"] / 68301
)

tbl4_index_class_distribution

In [ ]:
# Number of antidepressant classes at index date

class_count_order = [1, 2, 3, 4, 5]

number_of_classes_summary = (
    patient_index_class
    .groupby("n_class")["person_id"]
    .nunique()
    .reset_index()
    .rename(columns={
        "n_class": "number_of_classes_at_index_date",
        "person_id": "patient_n"
    })
)

# Force 1, 2, 3, 4, 5 to appear even if one category has 0 patients
number_of_classes_summary = (
    pd.DataFrame({"number_of_classes_at_index_date": class_count_order})
    .merge(
        number_of_classes_summary,
        on="number_of_classes_at_index_date",
        how="left"
    )
)

number_of_classes_summary["patient_n"] = (
    number_of_classes_summary["patient_n"]
    .fillna(0)
    .astype(int)
)

# Add percent column if you want to check, but you can copy only patient_n to Excel
number_of_classes_summary["percent_of_total"] = (
    number_of_classes_summary["patient_n"] / 68301
)

# Add total row
total_row = pd.DataFrame({
    "number_of_classes_at_index_date": ["Total"],
    "patient_n": [number_of_classes_summary["patient_n"].sum()],
    "percent_of_total": [number_of_classes_summary["patient_n"].sum() / 68301]
})

number_of_classes_summary = pd.concat(
    [number_of_classes_summary, total_row],
    ignore_index=True
)

number_of_classes_summary

In [ ]:
# Antidepressant class combination summary at index date
# Use the exact order requested in the Excel template

class_combo_order = [
    "Atypical Antidepressants + SSRI",
    "Atypical Antidepressants + SNRI",
    "SSRI + TCA",
    "SNRI + SSRI",
    "Atypical Antidepressants + TCA",
    "SNRI + TCA",
    "Atypical Antidepressants + MAOI",
    "MAOI + SSRI",
    "Atypical Antidepressants + SSRI + TCA",
    "Atypical Antidepressants + SNRI + SSRI",
    "Atypical Antidepressants + SNRI + TCA",
    "SNRI + SSRI + TCA",
    "Atypical Antidepressants + SNRI + SSRI + TCA",
    "Atypical Antidepressants + MAOI + SNRI + SSRI + TCA"
]

class_combo_summary = (
    patient_index_class
    .groupby("class_combo")["person_id"]
    .nunique()
    .reset_index()
    .rename(columns={"person_id": "patient_n"})
)

# Force the Excel order
class_combo_summary_ordered = (
    pd.DataFrame({"class_combo": class_combo_order})
    .merge(class_combo_summary, on="class_combo", how="left")
)

class_combo_summary_ordered["patient_n"] = (
    class_combo_summary_ordered["patient_n"]
    .fillna(0)
    .astype(int)
)

print("Sum of listed multiple-class combinations:",
      class_combo_summary_ordered["patient_n"].sum())

print("Multiple-class patients from patient_index_class:",
      (patient_index_class["n_class"] > 1).sum())

# Optional percent column
class_combo_summary_ordered["percent_of_total"] = (
    class_combo_summary_ordered["patient_n"] / 68301
)

class_combo_summary_ordered



In [ ]:
# Summary of generic drug combinations among multiple-index-class patients

multi_generic_summary = (
    patient_index_class[patient_index_class["n_class"] > 1]
    .groupby(["n_class", "class_combo", "generic_combo"], as_index=False)
    .agg(patient_n=("person_id", "nunique"))
)

# Sort by:
# 1. number of classes: 2, 3, 4, 5
# 2. class combination name
# 3. patient count from high to low within each class combination
multi_generic_summary = (
    multi_generic_summary
    .sort_values(
        by=["n_class", "class_combo", "patient_n", "generic_combo"],
        ascending=[True, True, False, True]
    )
    .reset_index(drop=True)
)

multi_generic_summary


In [ ]:
# Summary of generic drug combinations among multiple-index-class patients

multi_generic_summary = (
    patient_index_class[patient_index_class["n_class"] > 1]
    .groupby(["n_class", "class_combo", "generic_combo"], as_index=False)
    .agg(patient_n=("person_id", "nunique"))
)

# Desired class-combo order from the Excel template
class_combo_order = [
    "Atypical Antidepressants + SSRI",
    "Atypical Antidepressants + SNRI",
    "SSRI + TCA",
    "SNRI + SSRI",
    "Atypical Antidepressants + TCA",
    "SNRI + TCA",
    "Atypical Antidepressants + MAOI",
    "MAOI + SSRI",
    "Atypical Antidepressants + SSRI + TCA",
    "Atypical Antidepressants + SNRI + SSRI",
    "Atypical Antidepressants + SNRI + TCA",
    "SNRI + SSRI + TCA",
    "Atypical Antidepressants + SNRI + SSRI + TCA",
    "Atypical Antidepressants + MAOI + SNRI + SSRI + TCA"
]

class_combo_order_df = pd.DataFrame({
    "class_combo": class_combo_order,
    "combo_order": range(1, len(class_combo_order) + 1)
})

# Apply the Excel order, then sort generic combinations by patient count within each class combo
multi_generic_summary_ordered = (
    multi_generic_summary
    .merge(class_combo_order_df, on="class_combo", how="left")
    .sort_values(
        by=["combo_order", "patient_n", "generic_combo"],
        ascending=[True, False, True]
    )
    .drop(columns=["combo_order"])
    .reset_index(drop=True)
)

multi_generic_summary_ordered

In [ ]:
print("Sum of generic-combo patient_n:",
      multi_generic_summary_ordered["patient_n"].sum())

print("Multiple-class patients:",
      (patient_index_class["n_class"] > 1).sum())

In [ ]:
# Environment setup note: install package only if needed in a fresh environment.
# !pip install openpyxl

In [ ]:
# Environment setup note: install package only if needed in a fresh environment.
# pip install --upgrade pip

In [ ]:
# Export the full 312-row table to Excel
multi_generic_summary_ordered.to_excel(
    "Table4_multi_index_class_generic_combo_summary.xlsx",
    index=False
)

In [ ]:
output_file = "./Table4_multi_index_class_generic_combo_summary.xlsx"

multi_generic_summary_ordered.to_excel(output_file, index=False)

print("Saved to:", os.path.abspath(output_file))

In [ ]:
# Summary of generic drug combinations among single-index-class patients

single_generic_summary = (
    patient_index_class[patient_index_class["n_class"] == 1]
    .groupby(["class_combo", "generic_combo"], as_index=False)
    .agg(patient_n=("person_id", "nunique"))
)

# Add class order so output follows:
# SSRI, Atypical Antidepressants, SNRI, TCA, MAOI
class_order = {
    "SSRI": 1,
    "Atypical Antidepressants": 2,
    "SNRI": 3,
    "TCA": 4,
    "MAOI": 5
}

single_generic_summary["class_order"] = single_generic_summary["class_combo"].map(class_order)

# Sort by:
# 1. class order
# 2. patient count from high to low within each class
# 3. generic_combo alphabetically if same count
single_generic_summary = (
    single_generic_summary
    .sort_values(
        by=["class_order", "patient_n", "generic_combo"],
        ascending=[True, False, True]
    )
    .drop(columns=["class_order"])
    .reset_index(drop=True)
)

single_generic_summary



In [ ]:
print("Sum of patient_n:", single_generic_summary["patient_n"].sum())
print("Expected mono-class patients:", (patient_index_class["n_class"] == 1).sum())
print("Percent among 62448:", single_generic_summary["patient_n"].sum() / 62448)

In [ ]:
single_generic_summary.to_excel(
    "Table4_single_index_class_generic_combo_summary.xlsx",
    index=False
)

In [ ]:
# Summary of patients with only one generic drug at index date

mono_generic_summary = (
    patient_index_class[
        (patient_index_class["n_class"] == 1) &
        (patient_index_class["n_generic"] == 1)
    ]
    .groupby(["class_combo", "generic_combo"], as_index=False)
    .agg(patient_n=("person_id", "nunique"))
)

# Class order for sorting
class_order = {
    "TCA": 1,
    "SSRI": 2,
    "SNRI": 3,
    "MAOI": 4,
    "Atypical Antidepressants": 5
}

mono_generic_summary["class_order"] = mono_generic_summary["class_combo"].map(class_order)

# Sort by class order, then patient count from high to low
mono_generic_summary = (
    mono_generic_summary
    .sort_values(
        by=["class_order", "patient_n", "generic_combo"],
        ascending=[True, False, True]
    )
    .drop(columns=["class_order"])
    .reset_index(drop=True)
)

mono_generic_summary

In [ ]:
print("Mono-generic total:", mono_generic_summary["patient_n"].sum())

print(
    "Expected mono-generic patients:",
    ((patient_index_class["n_class"] == 1) & (patient_index_class["n_generic"] == 1)).sum()
)

In [ ]:
mono_generic_summary.to_excel(
    "Table4_mono_generic_at_index_date_summary.xlsx",
    index=False
)

## Table 5: Mono-class index-date cohort and duration analysis

This section focuses on mono-class index patients and evaluates index-date drug-use duration patterns.


In [ ]:
#table 5

In [ ]:
# Mono-class patients from Table 4
mono_class_patient_ids = patient_index_class.loc[
    patient_index_class["n_class"] == 1,
    "person_id"
]

print("Mono-class patients:", mono_class_patient_ids.nunique())

# Index-date drug exposure records for mono-class patients
mono_index_records = index_records[
    index_records["person_id"].isin(mono_class_patient_ids)
].copy()

print("Mono-class index-date patients:", mono_index_records["person_id"].nunique())
print("Mono-class index-date records:", len(mono_index_records))

# Convert start and end datetime to date only
mono_index_records["drug_exposure_start_datetime"] = pd.to_datetime(
    mono_index_records["drug_exposure_start_datetime"],
    errors="coerce",
    utc=True
)

mono_index_records["drug_exposure_end_datetime"] = pd.to_datetime(
    mono_index_records["drug_exposure_end_datetime"],
    errors="coerce",
    utc=True
)

mono_index_records["drug_start_date"] = mono_index_records["drug_exposure_start_datetime"].dt.date
mono_index_records["drug_end_date"] = mono_index_records["drug_exposure_end_datetime"].dt.date

# Record-level quality check
total_records = len(mono_index_records)
total_patients = mono_index_records["person_id"].nunique()

end_date_missing_n = mono_index_records["drug_end_date"].isna().sum()

end_date_earlier_than_start_n = (
    mono_index_records["drug_end_date"].notna()
    & (mono_index_records["drug_end_date"] < mono_index_records["drug_start_date"])
).sum()

record_level_qc = pd.DataFrame({
    "Metric": [
        "Total patients",
        "Total records",
        "End date missing",
        "End date earlier than start date"
    ],
    "Count": [
        total_patients,
        total_records,
        end_date_missing_n,
        end_date_earlier_than_start_n
    ]
})

record_level_qc["PCT"] = [
    total_patients / 62448,
    None,
    end_date_missing_n / total_records,
    end_date_earlier_than_start_n / total_records
]

record_level_qc


In [ ]:
# ------------------------------------------------------------
# Prepare datetime columns for Table 5 duration analysis
# ------------------------------------------------------------

# Original drug start/end datetime
mono_index_records["drug_exposure_start_datetime"] = pd.to_datetime(
    mono_index_records["drug_exposure_start_datetime"],
    errors="coerce",
    utc=True
)

mono_index_records["drug_exposure_end_datetime"] = pd.to_datetime(
    mono_index_records["drug_exposure_end_datetime"],
    errors="coerce",
    utc=True
)

# Convert to timezone-naive normalized datetime64 columns
# Do not use .dt.date here because groupby max() needs datetime64
mono_index_records["drug_start_date_dt"] = (
    mono_index_records["drug_exposure_start_datetime"]
    .dt.tz_convert(None)
    .dt.normalize()
)

mono_index_records["drug_end_date_dt"] = (
    mono_index_records["drug_exposure_end_datetime"]
    .dt.tz_convert(None)
    .dt.normalize()
)

# Convert index drug date to the same datetime64 format
mono_index_records["index_drug_date_dt"] = (
    pd.to_datetime(
        mono_index_records["index_drug_date"],
        errors="coerce",
        utc=True
    )
    .dt.tz_convert(None)
    .dt.normalize()
)

print(
    mono_index_records[
        [
            "index_drug_date_dt",
            "drug_start_date_dt",
            "drug_end_date_dt"
        ]
    ].dtypes
)

In [ ]:
# ------------------------------------------------------------
# RECORD LEVEL: INDEX DATE DRUG CLASS USAGE DURATION SUMMARY
# Using patient-level latest end date logic, then merged back to record level
# ------------------------------------------------------------

# Step 1: Get one latest end date per patient
patient_latest_end = (
    mono_index_records
    .groupby("person_id", as_index=False)
    .agg(
        index_drug_date=("index_drug_date_dt", "first"),
        latest_drug_end_date=("drug_end_date_dt", "max"),
        n_available_end_dates=("drug_end_date_dt", lambda x: x.notna().sum())
    )
)

# Step 2: Calculate patient-level duration
patient_latest_end["usage_duration_days"] = (
    patient_latest_end["latest_drug_end_date"]
    - patient_latest_end["index_drug_date"]
).dt.days

# If patient has no available end date among index-date records, set duration as missing
patient_latest_end.loc[
    patient_latest_end["n_available_end_dates"] == 0,
    "usage_duration_days"
] = np.nan

# Step 3: Merge patient-level duration back to record-level table
mono_index_records_with_patient_duration = mono_index_records.merge(
    patient_latest_end[["person_id", "latest_drug_end_date", "usage_duration_days"]],
    on="person_id",
    how="left"
)

# Step 4: Record-level counts
total_records = len(mono_index_records_with_patient_duration)

duration_missing_n = (
    mono_index_records_with_patient_duration["usage_duration_days"].isna()
).sum()

negative_duration_n = (
    mono_index_records_with_patient_duration["usage_duration_days"] < 0
).sum()

zero_duration_n = (
    mono_index_records_with_patient_duration["usage_duration_days"] == 0
).sum()

valid_duration = mono_index_records_with_patient_duration.loc[
    mono_index_records_with_patient_duration["usage_duration_days"].notna()
    & (mono_index_records_with_patient_duration["usage_duration_days"] >= 0),
    "usage_duration_days"
]

# Step 5: Record-level duration summary
record_level_usage_duration_summary = pd.DataFrame({
    "Metric": [
        "Total records",
        "duration_missing_n",
        "duration_missing_pct",
        "negative_duration_n",
        "negative_duration_pct",
        "zero_duration_n",
        "zero_duration_pct",
        "mean_duration",
        "sd_duration",
        "median_duration",
        "q1_duration",
        "q3_duration",
        "min_duration",
        "max_duration"
    ],
    "Value": [
        total_records,
        duration_missing_n,
        duration_missing_n / total_records,
        negative_duration_n,
        negative_duration_n / total_records,
        zero_duration_n,
        zero_duration_n / total_records,
        round(valid_duration.mean(), 1),
        round(valid_duration.std(), 1),
        round(valid_duration.median(), 1),
        round(valid_duration.quantile(0.25), 1),
        round(valid_duration.quantile(0.75), 1),
        int(valid_duration.min()),
        int(valid_duration.max())
    ]
})

record_level_usage_duration_summary


In [ ]:
# ============================================================
# VISUALIZATION: Record-level index drug class usage duration distribution
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

# Use the same valid duration records from the record-level summary
duration_plot_df = mono_index_records_with_patient_duration[
    mono_index_records_with_patient_duration["usage_duration_days"].notna()
    & (mono_index_records_with_patient_duration["usage_duration_days"] >= 0)
].copy()

duration_values = duration_plot_df["usage_duration_days"]

print("Number of valid duration records:", len(duration_values))
print("Mean:", round(duration_values.mean(), 1))
print("Median:", round(duration_values.median(), 1))
print("Q1:", round(duration_values.quantile(0.25), 1))
print("Q3:", round(duration_values.quantile(0.75), 1))
print("Min:", int(duration_values.min()))
print("Max:", int(duration_values.max()))

# Export histogram figure as PNG

import matplotlib.pyplot as plt
import os

duration_plot_df = mono_index_records_with_patient_duration[
    mono_index_records_with_patient_duration["usage_duration_days"].notna()
    & (mono_index_records_with_patient_duration["usage_duration_days"] >= 0)
].copy()

duration_values = duration_plot_df["usage_duration_days"]
duration_values_365 = duration_values[duration_values <= 365]

plt.figure(figsize=(9, 5))
plt.hist(duration_values_365, bins=50)

plt.xlabel("Usage duration days")
plt.ylabel("Number of drug records")
plt.title("Index-date antidepressant usage duration distribution, 0-365 days")

plt.axvline(
    duration_values.mean(),
    linestyle="--",
    label=f"Overall mean = {duration_values.mean():.1f} days"
)

plt.axvline(
    duration_values.median(),
    linestyle="--",
    label=f"Overall median = {duration_values.median():.1f} days"
)

plt.legend()
plt.tight_layout()

output_fig = "Table5_duration_distribution_0_365_days.png"

plt.savefig(output_fig, dpi=300, bbox_inches="tight")

plt.show()

print(os.path.abspath(output_fig))


In [ ]:
# Export full-range histogram figure as PNG

import matplotlib.pyplot as plt
import os

duration_plot_df = mono_index_records_with_patient_duration[
    mono_index_records_with_patient_duration["usage_duration_days"].notna()
    & (mono_index_records_with_patient_duration["usage_duration_days"] >= 0)
].copy()

duration_values = duration_plot_df["usage_duration_days"]

plt.figure(figsize=(9, 5))
plt.hist(duration_values, bins=100)

plt.xlabel("Usage duration days")
plt.ylabel("Number of drug records")
plt.title("Index-date antidepressant usage duration distribution, full range")

plt.axvline(
    duration_values.mean(),
    linestyle="--",
    label=f"Mean = {duration_values.mean():.1f} days"
)

plt.axvline(
    duration_values.median(),
    linestyle="--",
    label=f"Median = {duration_values.median():.1f} days"
)

plt.legend()
plt.tight_layout()

output_fig = "Table5_duration_distribution_full_range.png"

plt.savefig(output_fig, dpi=300, bbox_inches="tight")

plt.show()

print(os.path.abspath(output_fig))

In [ ]:
# ============================================================
# Export full-range log-transformed histogram as PNG
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import os

duration_plot_df = mono_index_records_with_patient_duration[
    mono_index_records_with_patient_duration["usage_duration_days"].notna()
    & (mono_index_records_with_patient_duration["usage_duration_days"] >= 0)
].copy()

duration_values = duration_plot_df["usage_duration_days"]

# Log transformation; +1 allows zero-duration records to be included
log_duration_values = np.log1p(duration_values)

plt.figure(figsize=(9, 5))

plt.hist(
    log_duration_values,
    bins=60
)

# Mean and median shown at their transformed positions
plt.axvline(
    np.log1p(duration_values.mean()),
    linestyle="--",
    label=f"Mean = {duration_values.mean():.1f} days"
)

plt.axvline(
    np.log1p(duration_values.median()),
    linestyle="--",
    label=f"Median = {duration_values.median():.1f} days"
)

plt.xlabel("log(Usage duration days + 1)")
plt.ylabel("Number of drug records")
plt.title(
    "Index-date antidepressant usage duration distribution, "
    "log-transformed full range"
)

plt.legend()
plt.tight_layout()

output_fig = "Table5_duration_distribution_log_full_range.png"

plt.savefig(
    output_fig,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(os.path.abspath(output_fig))

In [ ]:
# ------------------------------------------------------------
# Patient-level duration based on the footnote definition
# Duration = latest drug end date among all index-date records - index drug date
# Each patient contributes only one duration value
# Patients with no available end date at index are classified as missing
# ------------------------------------------------------------

# Make sure datetime columns are proper pandas datetime
mono_index_records["drug_exposure_start_datetime"] = pd.to_datetime(
    mono_index_records["drug_exposure_start_datetime"],
    errors="coerce",
    utc=True
)

mono_index_records["drug_exposure_end_datetime"] = pd.to_datetime(
    mono_index_records["drug_exposure_end_datetime"],
    errors="coerce",
    utc=True
)

# Use pandas datetime normalized to date-level, not Python .dt.date
mono_index_records["drug_start_date_dt"] = (
    mono_index_records["drug_exposure_start_datetime"]
    .dt.tz_convert(None)
    .dt.normalize()
)

mono_index_records["drug_end_date_dt"] = (
    mono_index_records["drug_exposure_end_datetime"]
    .dt.tz_convert(None)
    .dt.normalize()
)

mono_index_records["index_drug_date_dt"] = pd.to_datetime(
    mono_index_records["index_drug_date"],
    errors="coerce"
)

# Group to patient level
patient_level_duration = (
    mono_index_records
    .groupby("person_id", as_index=False)
    .agg(
        index_drug_date=("index_drug_date_dt", "first"),
        index_class=("ad_class", "first"),
        latest_drug_end_date=("drug_end_date_dt", "max"),
        n_index_records=("person_id", "size"),
        n_available_end_dates=("drug_end_date_dt", lambda x: x.notna().sum())
    )
)

# Calculate patient-level duration
patient_level_duration["patient_duration_days"] = (
    patient_level_duration["latest_drug_end_date"]
    - patient_level_duration["index_drug_date"]
).dt.days

# If no available end date among all index-date records, classify as missing
patient_level_duration.loc[
    patient_level_duration["n_available_end_dates"] == 0,
    "patient_duration_days"
] = np.nan

print("Patient-level total patients:", patient_level_duration["person_id"].nunique())
print("Patients with missing patient-level duration:", patient_level_duration["patient_duration_days"].isna().sum())
print("Patients with zero patient-level duration:", (patient_level_duration["patient_duration_days"] == 0).sum())

patient_level_duration.head()

In [ ]:
def duration_group(x):
    if pd.isna(x):
        return "Missing"
    if x < 0:
        return "Negative"
    if x == 0:
        return "0 days"
    if 1 <= x <= 7:
        return "1-7 days"
    if 8 <= x <= 30:
        return "8-30 days"
    if 31 <= x <= 90:
        return "31-90 days"
    if 91 <= x <= 180:
        return "91-180 days"
    if 181 <= x <= 365:
        return "181-365 days"
    if x > 365:
        return ">365 days"

patient_level_duration["duration_group"] = patient_level_duration[
    "patient_duration_days"
].apply(duration_group)

duration_group_order = [
    "0 days",
    "1-7 days",
    "8-30 days",
    "31-90 days",
    "91-180 days",
    "181-365 days",
    ">365 days",
    "Missing",
    "Negative"
]

patient_level_duration_distribution = (
    patient_level_duration
    .groupby("duration_group", as_index=False)
    .agg(patient_n=("person_id", "nunique"))
)

patient_level_duration_distribution = (
    pd.DataFrame({"duration_group": duration_group_order})
    .merge(patient_level_duration_distribution, on="duration_group", how="left")
)

patient_level_duration_distribution["patient_n"] = (
    patient_level_duration_distribution["patient_n"]
    .fillna(0)
    .astype(int)
)

patient_level_duration_distribution["pct_among_62448"] = (
    patient_level_duration_distribution["patient_n"] / 62448
)

patient_level_duration_distribution

In [ ]:
# ============================================================
# INVESTIGATE EXTREME PATIENT-LEVEL DURATIONS
# Patients with duration >= 3000 days
# ============================================================

extreme_duration_patients = (
    patient_level_duration[
        patient_level_duration["patient_duration_days"] >= 3000
    ]
    .sort_values(
        "patient_duration_days",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Patients with duration >= 3000 days:", len(extreme_duration_patients))

extreme_duration_patients[
    [
        "person_id",
        "index_drug_date",
        "latest_drug_end_date",
        "patient_duration_days",
        "index_class",
        "n_index_records",
        "n_available_end_dates"
    ]
].head(100)

In [ ]:
def extreme_duration_group(x):
    if x >= 8000:
        return "8000+ days"
    if x >= 7000:
        return "7000-7999 days"
    if x >= 6000:
        return "6000-6999 days"
    if x >= 5000:
        return "5000-5999 days"
    if x >= 4000:
        return "4000-4999 days"
    return "3000-3999 days"


extreme_duration_patients["extreme_duration_group"] = (
    extreme_duration_patients["patient_duration_days"]
    .apply(extreme_duration_group)
)

extreme_duration_group_summary = (
    extreme_duration_patients
    .groupby("extreme_duration_group", as_index=False)
    .agg(patient_n=("person_id", "nunique"))
)

group_order = [
    "8000+ days",
    "7000-7999 days",
    "6000-6999 days",
    "5000-5999 days",
    "4000-4999 days",
    "3000-3999 days"
]

extreme_duration_group_summary["extreme_duration_group"] = pd.Categorical(
    extreme_duration_group_summary["extreme_duration_group"],
    categories=group_order,
    ordered=True
)

extreme_duration_group_summary = (
    extreme_duration_group_summary
    .sort_values("extreme_duration_group")
    .reset_index(drop=True)
)

extreme_duration_group_summary

In [ ]:
# Prepare datetime columns safely

mono_index_records["drug_start_date_dt"] = (
    pd.to_datetime(
        mono_index_records["drug_exposure_start_datetime"],
        errors="coerce",
        utc=True
    )
    .dt.tz_convert(None)
    .dt.normalize()
)

mono_index_records["drug_end_date_dt"] = (
    pd.to_datetime(
        mono_index_records["drug_exposure_end_datetime"],
        errors="coerce",
        utc=True
    )
    .dt.tz_convert(None)
    .dt.normalize()
)

mono_index_records["index_drug_date_dt"] = (
    pd.to_datetime(
        mono_index_records["index_drug_date"],
        errors="coerce",
        utc=True
    )
    .dt.tz_convert(None)
    .dt.normalize()
)

In [ ]:
# All raw index-date drug records for extreme-duration patients

extreme_patient_ids = extreme_duration_patients["person_id"]

extreme_raw_records = (
    mono_index_records[
        mono_index_records["person_id"].isin(extreme_patient_ids)
    ]
    .merge(
        extreme_duration_patients[
            [
                "person_id",
                "index_drug_date",
                "latest_drug_end_date",
                "patient_duration_days",
                "extreme_duration_group"
            ]
        ],
        on="person_id",
        how="left",
        suffixes=("", "_patient")
    )
)

# Duration for each individual raw record
extreme_raw_records["individual_record_duration_days"] = (
    extreme_raw_records["drug_end_date_dt"]
    - extreme_raw_records["drug_start_date_dt"]
).dt.days

# Identify the record that supplied the patient's latest end date
extreme_raw_records["is_latest_end_date_record"] = (
    extreme_raw_records["drug_end_date_dt"]
    == pd.to_datetime(extreme_raw_records["latest_drug_end_date"])
)

extreme_raw_records = (
    extreme_raw_records
    .sort_values(
        [
            "patient_duration_days",
            "person_id",
            "is_latest_end_date_record",
            "drug_end_date_dt"
        ],
        ascending=[False, True, False, False]
    )
    .reset_index(drop=True)
)

In [ ]:
raw_columns_wanted = [
    "person_id",
    "extreme_duration_group",
    "patient_duration_days",
    "index_drug_date_dt",
    "drug_start_date_dt",
    "drug_end_date_dt",
    "individual_record_duration_days",
    "is_latest_end_date_record",
    "standard_concept_name",
    "ad_generic_name",
    "ad_class",
    "drug_type_concept_name",
    "days_supply",
    "refills",
    "quantity",
    "route_concept_name",
    "drug_concept_id",
    "standard_concept_code",
    "drug_source_value",
    "source_concept_name",
    "visit_occurrence_concept_name"
]

raw_columns_available = [
    col for col in raw_columns_wanted
    if col in extreme_raw_records.columns
]

extreme_raw_records_view = extreme_raw_records[
    raw_columns_available
]

extreme_raw_records_view.head(200)

In [ ]:
raw_columns_wanted = [
    "person_id",
    "extreme_duration_group",
    "patient_duration_days",
    "index_drug_date_dt",
    "drug_start_date_dt",
    "drug_end_date_dt",
    "individual_record_duration_days",
    "is_latest_end_date_record",
    "standard_concept_name",
    "ad_generic_name",
    "ad_class",
    "drug_type_concept_name",
    "days_supply",
    "refills",
    "quantity",
    "route_concept_name",
    "drug_concept_id",
    "standard_concept_code",
    "drug_source_value",
    "source_concept_name",
    "visit_occurrence_concept_name"
]

raw_columns_available = [
    col for col in raw_columns_wanted
    if col in extreme_raw_records.columns
]

extreme_raw_records_view = extreme_raw_records[
    raw_columns_available
]

extreme_raw_records_view.head(200)

In [ ]:
extreme_culprit_records = (
    extreme_raw_records[
        extreme_raw_records["is_latest_end_date_record"]
    ]
    [raw_columns_available]
    .sort_values(
        "patient_duration_days",
        ascending=False
    )
    .reset_index(drop=True)
)

extreme_culprit_records.head(200)
# Numeric versions
extreme_culprit_records["days_supply_numeric"] = pd.to_numeric(
    extreme_culprit_records.get("days_supply"),
    errors="coerce"
)

extreme_culprit_records["refills_numeric"] = pd.to_numeric(
    extreme_culprit_records.get("refills"),
    errors="coerce"
)

# Approximate potential coverage
extreme_culprit_records["estimated_supply_coverage_days"] = (
    extreme_culprit_records["days_supply_numeric"]
    * (
        extreme_culprit_records["refills_numeric"]
        .fillna(0) + 1
    )
)

# CDRv8 participant data cutoff used in your project
cdr_cutoff = pd.Timestamp("2023-10-01")

extreme_culprit_records["end_after_cdr_cutoff"] = (
    pd.to_datetime(extreme_culprit_records["drug_end_date_dt"])
    > cdr_cutoff
)

extreme_culprit_records["duration_over_10_years"] = (
    extreme_culprit_records["patient_duration_days"] > 3652
)

extreme_culprit_records["duration_over_20_years"] = (
    extreme_culprit_records["patient_duration_days"] > 7305
)

extreme_culprit_records["end_year"] = (
    pd.to_datetime(extreme_culprit_records["drug_end_date_dt"])
    .dt.year
)

extreme_culprit_records[
    [
        col for col in [
            "person_id",
            "index_drug_date_dt",
            "drug_end_date_dt",
            "end_year",
            "patient_duration_days",
            "standard_concept_name",
            "drug_type_concept_name",
            "days_supply_numeric",
            "refills_numeric",
            "estimated_supply_coverage_days",
            "end_after_cdr_cutoff",
            "duration_over_10_years",
            "duration_over_20_years"
        ]
        if col in extreme_culprit_records.columns
    ]
].head(200)

In [ ]:
extreme_end_year_summary = (
    extreme_culprit_records
    .groupby("end_year", dropna=False)
    .agg(
        patient_n=("person_id", "nunique"),
        record_n=("person_id", "size")
    )
    .reset_index()
    .sort_values("end_year")
)

extreme_end_year_summary

In [ ]:
extreme_end_date_summary = (
    extreme_culprit_records
    .groupby("drug_end_date_dt", dropna=False)
    .agg(
        patient_n=("person_id", "nunique")
    )
    .reset_index()
    .sort_values(
        ["patient_n", "drug_end_date_dt"],
        ascending=[False, True]
    )
)

extreme_end_date_summary.head(50)

In [ ]:
with pd.ExcelWriter(
    "Table5_extreme_duration_raw_data_investigation.xlsx"
) as writer:

    extreme_duration_patients.to_excel(
        writer,
        sheet_name="extreme_patients",
        index=False
    )

    extreme_culprit_records.to_excel(
        writer,
        sheet_name="latest_end_records",
        index=False
    )

    extreme_raw_records_view.to_excel(
        writer,
        sheet_name="all_raw_records",
        index=False
    )

    extreme_duration_group_summary.to_excel(
        writer,
        sheet_name="duration_groups",
        index=False
    )

    extreme_end_year_summary.to_excel(
        writer,
        sheet_name="end_year_summary",
        index=False
    )

    extreme_end_date_summary.to_excel(
        writer,
        sheet_name="end_date_summary",
        index=False
    )

## Data cutoff and recent-record checks

This section checks the latest available dates and record patterns for mono-class patients.


In [ ]:
# ============================================================
# CHECK DATA CUTOFF FOR THE 62,448 MONO-CLASS PATIENTS
# ============================================================

import pandas as pd

# 62,448 mono-class patient IDs
mono_ids = set(mono_class_patient_ids.dropna().unique())

# ------------------------------------------------------------
# Part 1. Check the latest date in index-date records
# ------------------------------------------------------------

index_date_check = mono_index_records.copy()

index_date_check["index_drug_date_check"] = pd.to_datetime(
    index_date_check["index_drug_date"],
    errors="coerce",
    utc=True
).dt.tz_convert(None)

index_date_check["drug_start_date_check"] = pd.to_datetime(
    index_date_check["drug_exposure_start_datetime"],
    errors="coerce",
    utc=True
).dt.tz_convert(None)

index_date_check["drug_end_date_check"] = pd.to_datetime(
    index_date_check["drug_exposure_end_datetime"],
    errors="coerce",
    utc=True
).dt.tz_convert(None)

print("Latest index drug date:")
print(index_date_check["index_drug_date_check"].max())

print("\nLatest drug start date among index records:")
print(index_date_check["drug_start_date_check"].max())

print("\nLatest drug end date among index records:")
print(index_date_check["drug_end_date_check"].max())

In [ ]:
# ------------------------------------------------------------
# Part 2. Check all antidepressant records for the 62,448 patients
# Replace ad_clean only if your full cleaned antidepressant
# dataframe has a different name
# ------------------------------------------------------------

mono_all_ad_records = ad_clean[
    ad_clean["person_id"].isin(mono_ids)
].copy()

mono_all_ad_records["drug_start_date_check"] = pd.to_datetime(
    mono_all_ad_records["drug_exposure_start_datetime"],
    errors="coerce",
    utc=True
).dt.tz_convert(None)

mono_all_ad_records["drug_end_date_check"] = pd.to_datetime(
    mono_all_ad_records["drug_exposure_end_datetime"],
    errors="coerce",
    utc=True
).dt.tz_convert(None)

latest_start_date = mono_all_ad_records["drug_start_date_check"].max()
latest_end_date = mono_all_ad_records["drug_end_date_check"].max()

print("Patients:", mono_all_ad_records["person_id"].nunique())
print("Total antidepressant records:", len(mono_all_ad_records))

print("\nLatest observed drug START date:")
print(latest_start_date)

print("\nLatest observed drug END date:")
print(latest_end_date)

In [ ]:
# Create date-only column
mono_all_ad_records["drug_start_date_only"] = (
    mono_all_ad_records["drug_start_date_check"].dt.date
)

october_2023_counts = (
    mono_all_ad_records[
        (mono_all_ad_records["drug_start_date_check"] >= "2023-10-01") &
        (mono_all_ad_records["drug_start_date_check"] < "2023-11-01")
    ]
    .groupby("drug_start_date_only")
    .agg(
        record_n=("person_id", "size"),
        patient_n=("person_id", "nunique")
    )
    .reset_index()
    .sort_values("drug_start_date_only")
)

october_2023_counts

## Table 6: LOT1 to LOT2 treatment-pattern analysis

This section constructs class-level episodes, defines LOT1 and LOT2, calculates overlap, and creates LOT transition summaries.


In [ ]:
#table 6


In [ ]:
# ============================================================
# TABLE 6
# Entire follow-up treatment episodes and class transitions
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# STEP 1. DEFINE DATA CUTOFF
# ============================================================

data_cutoff = pd.Timestamp("2023-10-01")


# ============================================================
# STEP 2. IDENTIFY THE 62,448 MONO-CLASS PATIENTS
# ============================================================

lot1_index_info = (
    patient_index_class.loc[
        patient_index_class["n_class"] == 1,
        [
            "person_id",
            "index_drug_date",
            "class_combo"
        ]
    ]
    .drop_duplicates(subset=["person_id"])
    .rename(
        columns={
            "class_combo": "lot1_class"
        }
    )
    .copy()
)

lot1_index_info["index_drug_date"] = (
    pd.to_datetime(
        lot1_index_info["index_drug_date"],
        errors="coerce",
        utc=True
    )
    .dt.tz_convert(None)
    .dt.normalize()
)

print(
    "Mono-class patients:",
    lot1_index_info["person_id"].nunique()
)
# ============================================================
# STEP 3. GET ALL ANTIDEPRESSANT RECORDS DURING FOLLOW-UP
# ============================================================

fu_records = (
    ad_clean[
        ad_clean["person_id"].isin(
            lot1_index_info["person_id"]
        )
    ]
    .copy()
)

# Add index date and LOT1 class
fu_records = fu_records.merge(
    lot1_index_info,
    on="person_id",
    how="inner"
)


# Convert start and end dates
fu_records["drug_start_date"] = (
    pd.to_datetime(
        fu_records["drug_exposure_start_datetime"],
        errors="coerce",
        utc=True
    )
    .dt.tz_convert(None)
    .dt.normalize()
)

fu_records["drug_end_date_original"] = (
    pd.to_datetime(
        fu_records["drug_exposure_end_datetime"],
        errors="coerce",
        utc=True
    )
    .dt.tz_convert(None)
    .dt.normalize()
)


# Keep records from index date through data cutoff
fu_records = fu_records[
    fu_records["drug_start_date"].notna()
    & (
        fu_records["drug_start_date"]
        >= fu_records["index_drug_date"]
    )
    & (
        fu_records["drug_start_date"]
        <= data_cutoff
    )
].copy()
# ============================================================
# STEP 4. IMPUTE MISSING END DATES
#
# Rule:
# Missing end date = drug start date + 30 days
# ============================================================

fu_records["end_date_imputed_flag"] = (
    fu_records["drug_end_date_original"].isna()
)

# Begin with original end date
fu_records["drug_end_date_clean"] = (
    fu_records["drug_end_date_original"].copy()
)

# Impute all missing end dates using 30 days
fu_records.loc[
    fu_records["end_date_imputed_flag"],
    "drug_end_date_clean"
] = (
    fu_records.loc[
        fu_records["end_date_imputed_flag"],
        "drug_start_date"
    ]
    + pd.Timedelta(days=30)
)

# Record source of clean end date
fu_records["end_date_source"] = "Original end date"

fu_records.loc[
    fu_records["end_date_imputed_flag"],
    "end_date_source"
] = "Imputed using 30 days"
# ============================================================
# STEP 5. DATA PREPARATION SUMMARY
# ============================================================

total_patients = fu_records["person_id"].nunique()
total_records = len(fu_records)

imputed_end_date_n = int(
    fu_records["end_date_imputed_flag"].sum()
)

remaining_missing_end_date_n = int(
    fu_records["drug_end_date_clean"]
    .isna()
    .sum()
)

clean_end_before_start_n = int(
    (
        fu_records["drug_end_date_clean"].notna()
        & (
            fu_records["drug_end_date_clean"]
            < fu_records["drug_start_date"]
        )
    ).sum()
)

table6_data_preparation_summary = pd.DataFrame({
    "Metric": [
        "Total patients",
        "Total records",
        "Imputed end date n in entire FU",
        "Imputed end date pct",
        "Remaining missing end date n",
        "Clean end date earlier than start date n",
        "Latest observed drug start date",
        "Latest original drug end date",
        "Latest clean drug end date"
    ],
    "Value": [
        total_patients,
        total_records,
        imputed_end_date_n,
        imputed_end_date_n / total_records,
        remaining_missing_end_date_n,
        clean_end_before_start_n,
        fu_records["drug_start_date"].max(),
        fu_records["drug_end_date_original"].max(),
        fu_records["drug_end_date_clean"].max()
    ]
})

table6_data_preparation_summary

In [ ]:
# ============================================================
# STEP 6. CONSOLIDATE RECORDS INTO CLASS-LEVEL EPISODES
#
# Same patient + same antidepressant class:
# gap < 30 days  -> same episode
# gap >= 30 days -> new episode
# ============================================================

episode_input = fu_records[
    fu_records["ad_class"].notna()
    & fu_records["drug_start_date"].notna()
    & fu_records["drug_end_date_clean"].notna()
].copy()

episode_input = (
    episode_input
    .sort_values(
        [
            "person_id",
            "ad_class",
            "drug_start_date",
            "drug_end_date_clean"
        ]
    )
    .reset_index(drop=True)
)


# Running maximum end date within each patient/class
episode_input["running_max_end"] = (
    episode_input
    .groupby(
        ["person_id", "ad_class"]
    )["drug_end_date_clean"]
    .cummax()
)

# Maximum end date before the current record
episode_input["previous_max_end"] = (
    episode_input
    .groupby(
        ["person_id", "ad_class"]
    )["running_max_end"]
    .shift(1)
)


# Gap between current record start and previous episode coverage
episode_input["gap_days"] = (
    episode_input["drug_start_date"]
    - episode_input["previous_max_end"]
).dt.days


# New episode if first record or gap >= 30 days
episode_input["new_episode_flag"] = (
    episode_input["previous_max_end"].isna()
    | (
        episode_input["gap_days"] >= 30
    )
)


episode_input["episode_number_within_class"] = (
    episode_input
    .groupby(
        ["person_id", "ad_class"]
    )["new_episode_flag"]
    .cumsum()
    .astype(int)
)
# Aggregate records into episodes

class_episodes = (
    episode_input
    .groupby(
        [
            "person_id",
            "ad_class",
            "episode_number_within_class"
        ],
        as_index=False
    )
    .agg(
        episode_start=(
            "drug_start_date",
            "min"
        ),
        episode_end=(
            "drug_end_date_clean",
            "max"
        ),
        episode_record_n=(
            "person_id",
            "size"
        ),
        imputed_end_date_n=(
            "end_date_imputed_flag",
            "sum"
        ),
        generic_n=(
            "ad_generic_name",
            lambda x: x.dropna().nunique()
        ),
        generic_combo=(
            "ad_generic_name",
            lambda x: " + ".join(
                sorted(
                    set(
                        x.dropna().astype(str)
                    )
                )
            )
        )
    )
)


class_episodes["episode_duration_days"] = (
    class_episodes["episode_end"]
    - class_episodes["episode_start"]
).dt.days


class_episodes = (
    class_episodes
    .sort_values(
        [
            "person_id",
            "episode_start",
            "episode_end",
            "ad_class"
        ]
    )
    .reset_index(drop=True)
)


print(
    "Patients:",
    class_episodes["person_id"].nunique()
)

print(
    "Original records:",
    len(episode_input)
)

print(
    "Class-level episodes:",
    len(class_episodes)
)

class_episodes.head(20)

In [ ]:
episode_consolidation_summary = pd.DataFrame({
    "Metric": [
        "Patients",
        "Original antidepressant records",
        "Class-level episodes",
        "Records merged into existing episodes",
        "Mean records per episode",
        "Median records per episode",
        "Maximum records in one episode"
    ],
    "Value": [
        class_episodes["person_id"].nunique(),
        len(episode_input),
        len(class_episodes),
        len(episode_input) - len(class_episodes),
        round(
            class_episodes["episode_record_n"].mean(),
            2
        ),
        class_episodes["episode_record_n"].median(),
        class_episodes["episode_record_n"].max()
    ]
})

episode_consolidation_summary

In [ ]:
# ============================================================
# STEP 7. IDENTIFY LOT1 EPISODE
#
# LOT1:
# episode class = index class
# episode contains the index drug date
# ============================================================

lot1_candidates = class_episodes.merge(
    lot1_index_info,
    on="person_id",
    how="inner"
)


lot1_candidates = lot1_candidates[
    (
        lot1_candidates["ad_class"]
        == lot1_candidates["lot1_class"]
    )
    & (
        lot1_candidates["episode_start"]
        <= lot1_candidates["index_drug_date"]
    )
    & (
        lot1_candidates["episode_end"]
        >= lot1_candidates["index_drug_date"]
    )
].copy()

lot1_episodes = (
    lot1_candidates
    .sort_values(
        [
            "person_id",
            "episode_start",
            "episode_end"
        ],
        ascending=[True, True, False]
    )
    .drop_duplicates(
        subset=["person_id"],
        keep="first"
    )
    .rename(
        columns={
            "episode_start": "lot1_start",
            "episode_end": "lot1_end",
            "episode_record_n": "lot1_record_n",
            "episode_duration_days": "lot1_duration_days",
            "generic_n": "lot1_generic_n",
            "generic_combo": "lot1_generic_combo"
        }
    )
    [
        [
            "person_id",
            "index_drug_date",
            "lot1_class",
            "lot1_start",
            "lot1_end",
            "lot1_duration_days",
            "lot1_record_n",
            "lot1_generic_n",
            "lot1_generic_combo"
        ]
    ]
    .reset_index(drop=True)
)


print(
    "Patients with identified LOT1:",
    lot1_episodes["person_id"].nunique()
)

print(
    "Patients missing LOT1:",
    lot1_index_info["person_id"].nunique()
    - lot1_episodes["person_id"].nunique()
)

lot1_episodes.head()

In [ ]:
# ============================================================
# STEP 8. IDENTIFY SUBSEQUENT DIFFERENT-CLASS EPISODES
# ============================================================

episodes_with_lot1 = class_episodes.merge(
    lot1_episodes[
        [
            "person_id",
            "index_drug_date",
            "lot1_class",
            "lot1_start",
            "lot1_end"
        ]
    ],
    on="person_id",
    how="inner"
)


patient_class_transition = (
    episodes_with_lot1[
        (
            episodes_with_lot1["episode_start"]
            > episodes_with_lot1["index_drug_date"]
        )
        & (
            episodes_with_lot1["ad_class"]
            != episodes_with_lot1["lot1_class"]
        )
    ]
    .copy()
)


patient_class_transition = (
    patient_class_transition
    .rename(
        columns={
            "ad_class": "transition_class",
            "episode_start": "transition_start",
            "episode_end": "transition_end",
            "episode_record_n": "transition_record_n",
            "generic_combo": "transition_generic_combo"
        }
    )
)

In [ ]:
# ============================================================
# STEP 9. CALCULATE INCLUSIVE OVERLAP
# ============================================================

# Later of the two start dates
patient_class_transition["overlap_start"] = (
    patient_class_transition[
        [
            "lot1_start",
            "transition_start"
        ]
    ]
    .max(axis=1)
)

# Earlier of the two end dates
patient_class_transition["overlap_end"] = (
    patient_class_transition[
        [
            "lot1_end",
            "transition_end"
        ]
    ]
    .min(axis=1)
)


# Inclusive overlap:
# overlap end - overlap start + 1
patient_class_transition["raw_overlap_days"] = (
    patient_class_transition["overlap_end"]
    - patient_class_transition["overlap_start"]
).dt.days + 1


# No overlap or a gap becomes zero
patient_class_transition["overlap_days"] = (
    patient_class_transition["raw_overlap_days"]
    .clip(lower=0)
    .astype(int)
)


# Optional gap measure
patient_class_transition["gap_days"] = (
    patient_class_transition["transition_start"]
    - patient_class_transition["lot1_end"]
).dt.days.clip(lower=0)

In [ ]:
# ============================================================
# STEP 10. CLASSIFY OVERLAP
# ============================================================

patient_class_transition["overlap_category"] = np.select(
    [
        patient_class_transition["overlap_days"] == 0,

        patient_class_transition["overlap_days"].between(
            1, 29
        ),

        patient_class_transition["overlap_days"].between(
            30, 89
        ),

        patient_class_transition["overlap_days"] >= 90
    ],
    [
        "0 days",
        "1-29 days",
        "30-89 days",
        ">=90 days"
    ],
    default="Check"
)

In [ ]:
# ============================================================
# STEP 11. TRANSITION-EVENT SUMMARY
# ============================================================

overlap_order = [
    "0 days",
    "1-29 days",
    "30-89 days",
    ">=90 days"
]


transition_summary = (
    patient_class_transition
    .groupby(
        "overlap_category",
        as_index=False
    )
    .agg(
        transition_n=(
            "person_id",
            "size"
        ),
        patient_n=(
            "person_id",
            "nunique"
        )
    )
)


transition_summary = (
    pd.DataFrame({
        "overlap_category": overlap_order
    })
    .merge(
        transition_summary,
        on="overlap_category",
        how="left"
    )
)


transition_summary[
    [
        "transition_n",
        "patient_n"
    ]
] = (
    transition_summary[
        [
            "transition_n",
            "patient_n"
        ]
    ]
    .fillna(0)
    .astype(int)
)


total_transition_n = (
    transition_summary["transition_n"].sum()
)


transition_summary["transition_pct"] = (
    transition_summary["transition_n"]
    / total_transition_n
)


transition_summary

In [ ]:
# ============================================================
# TABLE 6 - AFTER TRANSITION SUMMARY
# Fixed display order and helper functions
# ============================================================

import pandas as pd
import numpy as np


# LOT1 fixed order, matching the teacher's table
lot1_class_order = [
    "SSRI",
    "Atypical Antidepressants",
    "SNRI",
    "TCA",
    "MAOI"
]

lot1_rank = {
    class_name: rank
    for rank, class_name in enumerate(lot1_class_order)
}


# LOT2 display order observed in the teacher's table
lot2_class_order = [
    "Atypical Antidepressants",
    "MAOI",
    "SNRI",
    "SSRI",
    "TCA"
]

lot2_rank = {
    class_name: rank
    for rank, class_name in enumerate(lot2_class_order)
}


# Overlap category order
overlap_category_order = [
    "Short / No overlap (<90 days)",
    "Long overlap (>=90 days)",
    "No qualifying non-index class in window"
]

overlap_rank = {
    category: rank
    for rank, category in enumerate(overlap_category_order)
}


def make_ordered_lot2_combo(class_series):
    """
    If multiple non-index classes start on the same earliest
    LOT2 date, combine them in a fixed class order.
    """

    unique_classes = list(
        set(class_series.dropna().astype(str))
    )

    unique_classes = sorted(
        unique_classes,
        key=lambda x: lot2_rank.get(x, 999)
    )

    return " + ".join(unique_classes)


def make_lot2_sort_key(lot2_label):
    """
    Create a sorting key for single and combined LOT2 classes.
    """

    if pd.isna(lot2_label):
        return (999, 999, (999,))

    components = [
        component.strip()
        for component in str(lot2_label).split(" + ")
    ]

    component_ranks = tuple(
        sorted(
            lot2_rank.get(component, 999)
            for component in components
        )
    )

    return (
        component_ranks[0],
        len(component_ranks),
        component_ranks
    )

In [ ]:
# ============================================================
# Build patient-level LOT details and complete frequency table
# ============================================================

def build_lot_tables(
    class_episodes_df,
    lot1_episodes_df,
    window_name,
    followup_days=None,
    recent_years=None,
    data_cutoff=None
):
    """
    Build three outputs for one analysis window:

    1. patient_detail:
       One row per eligible patient.

    2. lot_summary:
       Complete LOT1 -> earliest LOT2 frequency table.

    3. lot1_check:
       Patient counts by LOT1 class.

    Parameters
    ----------
    followup_days:
        365 for 1-year FU
        730 for 2-year FU
        None for entire FU

    recent_years:
        5 for recent 5-year cohort
        None otherwise
    """

    if data_cutoff is None:
        raise ValueError("data_cutoff must be provided.")


    # --------------------------------------------------------
    # 1. Establish eligible patients
    # --------------------------------------------------------

    base_patients = lot1_episodes_df[
        [
            "person_id",
            "index_drug_date",
            "lot1_class",
            "lot1_start",
            "lot1_end"
        ]
    ].copy()


    # Recent 5-year cohort:
    # restrict based on each patient's index date
    if recent_years is not None:

        recent_start_date = (
            data_cutoff
            - pd.DateOffset(years=recent_years)
        )

        base_patients = base_patients[
            (
                base_patients["index_drug_date"]
                >= recent_start_date
            )
            & (
                base_patients["index_drug_date"]
                <= data_cutoff
            )
        ].copy()


    denominator_n = (
        base_patients["person_id"].nunique()
    )


    # --------------------------------------------------------
    # 2. Attach all class-level episodes
    # --------------------------------------------------------

    lot2_candidates = class_episodes_df.merge(
        base_patients,
        on="person_id",
        how="inner"
    )


    # Qualifying LOT2:
    # post-index and different from LOT1 class
    lot2_candidates = lot2_candidates[
        (
            lot2_candidates["episode_start"]
            > lot2_candidates["index_drug_date"]
        )
        & (
            lot2_candidates["ad_class"]
            != lot2_candidates["lot1_class"]
        )
        & (
            lot2_candidates["episode_start"]
            <= data_cutoff
        )
    ].copy()


    # --------------------------------------------------------
    # 3. Apply 1-year or 2-year post-index window
    # --------------------------------------------------------

    if followup_days is not None:

        lot2_candidates["followup_window_end"] = (
            lot2_candidates["index_drug_date"]
            + pd.to_timedelta(
                followup_days,
                unit="D"
            )
        )

        lot2_candidates = lot2_candidates[
            (
                lot2_candidates["episode_start"]
                <= lot2_candidates["followup_window_end"]
            )
        ].copy()


    # --------------------------------------------------------
    # 4. Calculate overlap for every candidate episode
    # --------------------------------------------------------

    lot2_candidates["overlap_start"] = (
        lot2_candidates[
            [
                "lot1_start",
                "episode_start"
            ]
        ]
        .max(axis=1)
    )

    lot2_candidates["overlap_end"] = (
        lot2_candidates[
            [
                "lot1_end",
                "episode_end"
            ]
        ]
        .min(axis=1)
    )


    # Inclusive overlap:
    # overlap_end - overlap_start + 1
    lot2_candidates["overlap_days"] = (
        (
            lot2_candidates["overlap_end"]
            - lot2_candidates["overlap_start"]
        ).dt.days
        + 1
    )

    lot2_candidates["overlap_days"] = (
        lot2_candidates["overlap_days"]
        .clip(lower=0)
        .astype(int)
    )


    # --------------------------------------------------------
    # 5. Find earliest qualifying LOT2 date per patient
    # --------------------------------------------------------

    earliest_lot2_date = (
        lot2_candidates
        .groupby(
            "person_id",
            as_index=False
        )
        .agg(
            lot2_start=("episode_start", "min")
        )
    )


    # Keep all classes beginning on the same earliest date
    earliest_lot2_records = lot2_candidates.merge(
        earliest_lot2_date,
        left_on=[
            "person_id",
            "episode_start"
        ],
        right_on=[
            "person_id",
            "lot2_start"
        ],
        how="inner"
    )


    # --------------------------------------------------------
    # 6. Collapse same-day LOT2 classes to one patient row
    # --------------------------------------------------------

    patient_lot2 = (
        earliest_lot2_records
        .groupby(
            "person_id",
            as_index=False
        )
        .agg(
            lot2_start=(
                "lot2_start",
                "first"
            ),

            lot2_class=(
                "ad_class",
                make_ordered_lot2_combo
            ),

            lot2_class_n=(
                "ad_class",
                lambda x: x.dropna().nunique()
            ),

            # Footnote rule:
            # maximum overlap across same-day LOT2 classes
            overlap_days=(
                "overlap_days",
                "max"
            ),

            lot2_episode_end=(
                "episode_end",
                "max"
            )
        )
    )


    # --------------------------------------------------------
    # 7. Merge LOT2 into all eligible patients
    # --------------------------------------------------------

    patient_detail = base_patients.merge(
        patient_lot2,
        on="person_id",
        how="left"
    )


    no_qualifying_lot2 = (
        patient_detail["lot2_class"].isna()
    )


    # Match the teacher's table:
    # repeat LOT1 under LOT2 when no qualifying class exists
    patient_detail.loc[
        no_qualifying_lot2,
        "lot2_class"
    ] = patient_detail.loc[
        no_qualifying_lot2,
        "lot1_class"
    ]


    # --------------------------------------------------------
    # 8. Assign overlap category
    # --------------------------------------------------------

    patient_detail["overlap_category"] = np.select(
        [
            no_qualifying_lot2,

            patient_detail["overlap_days"] >= 90,

            patient_detail["overlap_days"] < 90
        ],
        [
            "No qualifying non-index class in window",

            "Long overlap (>=90 days)",

            "Short / No overlap (<90 days)"
        ],
        default="Check"
    )


    patient_detail["analysis_window"] = window_name


    # --------------------------------------------------------
    # 9. Patient-level detail table
    # --------------------------------------------------------

    patient_detail = patient_detail[
        [
            "person_id",
            "analysis_window",
            "index_drug_date",
            "lot1_class",
            "lot1_start",
            "lot1_end",
            "lot2_class",
            "lot2_class_n",
            "lot2_start",
            "lot2_episode_end",
            "overlap_days",
            "overlap_category"
        ]
    ].copy()


    patient_detail["lot1_order"] = (
        patient_detail["lot1_class"]
        .map(lot1_rank)
    )

    patient_detail["lot2_order"] = (
        patient_detail["lot2_class"]
        .apply(make_lot2_sort_key)
    )

    patient_detail["overlap_order"] = (
        patient_detail["overlap_category"]
        .map(overlap_rank)
    )


    patient_detail = (
        patient_detail
        .sort_values(
            [
                "lot1_order",
                "lot2_order",
                "overlap_order",
                "person_id"
            ]
        )
        .drop(
            columns=[
                "lot1_order",
                "lot2_order",
                "overlap_order"
            ]
        )
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # 10. Complete frequency summary
    # --------------------------------------------------------

    lot_summary = (
        patient_detail
        .groupby(
            [
                "lot1_class",
                "lot2_class",
                "overlap_category"
            ],
            as_index=False
        )
        .agg(
            patient_n=("person_id", "nunique")
        )
    )


    lot_summary["pct"] = (
        lot_summary["patient_n"]
        / denominator_n
    )


    lot_summary["lot1_order"] = (
        lot_summary["lot1_class"]
        .map(lot1_rank)
    )

    lot_summary["lot2_order"] = (
        lot_summary["lot2_class"]
        .apply(make_lot2_sort_key)
    )

    lot_summary["overlap_order"] = (
        lot_summary["overlap_category"]
        .map(overlap_rank)
    )


    lot_summary = (
        lot_summary
        .sort_values(
            [
                "lot1_order",
                "lot2_order",
                "overlap_order"
            ]
        )
        .drop(
            columns=[
                "lot1_order",
                "lot2_order",
                "overlap_order"
            ]
        )
        .reset_index(drop=True)
    )


    lot_summary.insert(
        0,
        "analysis_window",
        window_name
    )


    lot_summary = lot_summary[
        [
            "analysis_window",
            "lot1_class",
            "lot2_class",
            "overlap_category",
            "patient_n",
            "pct"
        ]
    ]


    # --------------------------------------------------------
    # 11. LOT1 patient-count check
    # --------------------------------------------------------

    lot1_check = (
        patient_detail
        .groupby(
            "lot1_class",
            as_index=False
        )
        .agg(
            patient_n=("person_id", "nunique")
        )
    )


    lot1_check["lot1_order"] = (
        lot1_check["lot1_class"]
        .map(lot1_rank)
    )


    lot1_check = (
        lot1_check
        .sort_values("lot1_order")
        .drop(columns="lot1_order")
        .reset_index(drop=True)
    )


    lot1_check["pct"] = (
        lot1_check["patient_n"]
        / denominator_n
    )


    total_row = pd.DataFrame({
        "lot1_class": ["Total"],
        "patient_n": [
            lot1_check["patient_n"].sum()
        ],
        "pct": [1.0]
    })


    lot1_check = pd.concat(
        [
            lot1_check,
            total_row
        ],
        ignore_index=True
    )


    # --------------------------------------------------------
    # 12. Print validation
    # --------------------------------------------------------

    print("=" * 60)
    print("Window:", window_name)
    print("Eligible patients:", denominator_n)
    print(
        "Patient-detail rows:",
        len(patient_detail)
    )
    print(
        "Summary patient total:",
        lot_summary["patient_n"].sum()
    )
    print("=" * 60)


    return (
        patient_detail,
        lot_summary,
        lot1_check
    )

In [ ]:
# ============================================================
# 1-YEAR FOLLOW-UP
# ============================================================

patient_lot_1y, lot_summary_1y, lot1_check_1y = (
    build_lot_tables(
        class_episodes_df=class_episodes,
        lot1_episodes_df=lot1_episodes,
        window_name="1-year FU",
        followup_days=365,
        recent_years=None,
        data_cutoff=data_cutoff
    )
)
patient_lot_1y.head(30)

In [ ]:
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None
):
    display(lot_summary_1y)

In [ ]:
# ============================================================
# 2-YEAR FOLLOW-UP
# ============================================================

patient_lot_2y, lot_summary_2y, lot1_check_2y = (
    build_lot_tables(
        class_episodes_df=class_episodes,
        lot1_episodes_df=lot1_episodes,
        window_name="2-year FU",
        followup_days=730,
        recent_years=None,
        data_cutoff=data_cutoff
    )
)
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None
):
    display(lot_summary_2y)

In [ ]:
# ============================================================
# ENTIRE FOLLOW-UP
# ============================================================

patient_lot_entire, lot_summary_entire, lot1_check_entire = (
    build_lot_tables(
        class_episodes_df=class_episodes,
        lot1_episodes_df=lot1_episodes,
        window_name="Entire FU",
        followup_days=None,
        recent_years=None,
        data_cutoff=data_cutoff
    )
)
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None
):
    display(lot_summary_entire)

In [ ]:
# ============================================================
# RECENT 5-YEAR COHORT
# ============================================================

patient_lot_recent5y, lot_summary_recent5y, lot1_check_recent5y = (
    build_lot_tables(
        class_episodes_df=class_episodes,
        lot1_episodes_df=lot1_episodes,
        window_name="Recent 5-year cohort",
        followup_days=None,
        recent_years=5,
        data_cutoff=data_cutoff
    )
)
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None
):
    display(lot_summary_recent5y)

In [ ]:
# ============================================================
# VALIDATION CHECK
# ============================================================

lot_summary_validation = pd.DataFrame({
    "table": [
        "lot_summary_1y",
        "lot_summary_2y",
        "lot_summary_entire",
        "lot_summary_recent5y"
    ],
    "patient_total": [
        lot_summary_1y["patient_n"].sum(),
        lot_summary_2y["patient_n"].sum(),
        lot_summary_entire["patient_n"].sum(),
        lot_summary_recent5y["patient_n"].sum()
    ],
    "pct_total": [
        lot_summary_1y["pct"].sum(),
        lot_summary_2y["pct"].sum(),
        lot_summary_entire["pct"].sum(),
        lot_summary_recent5y["pct"].sum()
    ]
})

lot_summary_validation
# ============================================================
# EXPORT ALL TABLE 6 LOT RESULTS TO EXCEL
# ============================================================

output_path = (
    "/home/jupyter/workspaces/"
    "mddprojectcontrolledtier/"
    "Table6_LOT_summary_results.xlsx"
)


with pd.ExcelWriter(
    output_path,
    engine="openpyxl"
) as writer:

    # Complete summaries
    lot_summary_1y.to_excel(
        writer,
        sheet_name="lot_summary_1y",
        index=False
    )

    lot_summary_2y.to_excel(
        writer,
        sheet_name="lot_summary_2y",
        index=False
    )

    lot_summary_entire.to_excel(
        writer,
        sheet_name="lot_summary_entire",
        index=False
    )

    lot_summary_recent5y.to_excel(
        writer,
        sheet_name="lot_summary_recent5y",
        index=False
    )


    # LOT1 count checks
    lot1_check_1y.to_excel(
        writer,
        sheet_name="lot1_check_1y",
        index=False
    )

    lot1_check_2y.to_excel(
        writer,
        sheet_name="lot1_check_2y",
        index=False
    )

    lot1_check_entire.to_excel(
        writer,
        sheet_name="lot1_check_entire",
        index=False
    )

    lot1_check_recent5y.to_excel(
        writer,
        sheet_name="lot1_check_recent5y",
        index=False
    )


    # Patient-level details
    patient_lot_1y.to_excel(
        writer,
        sheet_name="patient_lot_1y",
        index=False
    )

    patient_lot_2y.to_excel(
        writer,
        sheet_name="patient_lot_2y",
        index=False
    )

    patient_lot_entire.to_excel(
        writer,
        sheet_name="patient_lot_entire",
        index=False
    )

    patient_lot_recent5y.to_excel(
        writer,
        sheet_name="patient_lot_recent5y",
        index=False
    )


    # Validation
    lot_summary_validation.to_excel(
        writer,
        sheet_name="validation",
        index=False
    )


print("Exported to:")
print(output_path)

## Table 6 visualizations: Sankey plots

This section creates Sankey plots for LOT1 to LOT2 treatment-pattern flows across follow-up windows.


In [ ]:
# ============================================================
# TABLE 6 - SANKEY PLOTS
# LOT1 -> LOT2 -> OVERLAP CATEGORY
# ============================================================

import os
import pandas as pd
import plotly.graph_objects as go


# Output folder
output_dir = "/home/jupyter/workspaces/mddprojectcontrolledtier"

os.makedirs(
    output_dir,
    exist_ok=True
)


# LOT1 fixed order
lot1_order = [
    "SSRI",
    "Atypical Antidepressants",
    "SNRI",
    "TCA",
    "MAOI"
]


# LOT2 base order matching the teacher's table
lot2_base_order = [
    "Atypical Antidepressants",
    "MAOI",
    "SNRI",
    "SSRI",
    "TCA"
]


# Overlap category order
overlap_order = [
    "Short / No overlap (<90 days)",
    "Long overlap (>=90 days)",
    "No qualifying non-index class in window"
]


lot1_rank = {
    name: i
    for i, name in enumerate(lot1_order)
}

lot2_rank = {
    name: i
    for i, name in enumerate(lot2_base_order)
}

overlap_rank = {
    name: i
    for i, name in enumerate(overlap_order)
}

In [ ]:
overlap_rank = {
    name: i
    for i, name in enumerate(overlap_order)
}


def make_lot_sankey_footnote(window_label):
    """
    Create explanatory footnote for LOT Sankey plots.
    Line breaks are added for better readability in HTML/PNG outputs.
    """

    return (
        "<b>Note.</b> LOT = line of therapy.<br>"
        "LOT1 represents the first observed antidepressant treatment class after the index MDD diagnosis. "
        f"LOT2 represents the first qualifying subsequent non-index antidepressant class identified within the {window_label}.<br>"
        "Sankey flow width represents patient count, not prescription count or drug exposure count. "
        "Percentages shown in hover labels are calculated using the total N for the current plot.<br>"
        "Overlap categories describe overlapping exposure between LOT1 and LOT2: "
        "short/no overlap is &lt;90 days and long overlap is ≥90 days.<br>"
        "The no qualifying non-index class category indicates that no subsequent antidepressant class meeting LOT2 criteria was identified within the specified window.<br>"
        "Antidepressant classes include SSRI, SNRI, atypical antidepressants, TCA, and MAOI."
    )

In [ ]:
def get_lot2_sort_key(lot2_label):
    """
    Sort single and combined LOT2 classes according to
    the teacher's class order.
    """

    components = [
        item.strip()
        for item in str(lot2_label).split(" + ")
    ]

    component_ranks = tuple(
        lot2_rank.get(item, 999)
        for item in components
    )

    return (
        min(component_ranks),
        len(components),
        component_ranks
    )

In [ ]:
def create_lot_sankey(
    summary_df,
    plot_title,
    file_name,
    footnote_text=None
):
    """
    Create a three-level Sankey plot:

    LOT1 -> LOT2 -> overlap category

    Parameters
    ----------
    summary_df:
        One of:
        lot_summary_1y
        lot_summary_2y
        lot_summary_entire
        lot_summary_recent5y

    plot_title:
        Title shown on the plot.

    file_name:
        Base filename without extension.
    """

    df = summary_df.copy()

    # Ensure patient counts are numeric
    df["patient_n"] = pd.to_numeric(
        df["patient_n"],
        errors="coerce"
    ).fillna(0).astype(int)

    # Calculate denominator directly from this summary
    denominator_n = int(
        df["patient_n"].sum()
    )

    # Recalculate percentage for hover labels
    df["plot_pct"] = (
        df["patient_n"]
        / denominator_n
        * 100
    )


    # --------------------------------------------------------
    # 1. Sort rows
    # --------------------------------------------------------

    df["lot1_sort"] = (
        df["lot1_class"]
        .map(lot1_rank)
        .fillna(999)
    )

    df["lot2_sort"] = (
        df["lot2_class"]
        .apply(get_lot2_sort_key)
    )

    df["overlap_sort"] = (
        df["overlap_category"]
        .map(overlap_rank)
        .fillna(999)
    )

    df = (
        df
        .sort_values(
            [
                "lot1_sort",
                "lot2_sort",
                "overlap_sort"
            ]
        )
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # 2. Create unique node keys
    #
    # LOT2 node is made unique within each LOT1 block.
    # This prevents different LOT1 pathways from mixing.
    # --------------------------------------------------------

    df["lot1_node_key"] = (
        "LOT1||"
        + df["lot1_class"].astype(str)
    )

    df["lot2_node_key"] = (
        "LOT2||"
        + df["lot1_class"].astype(str)
        + "||"
        + df["lot2_class"].astype(str)
    )

    df["overlap_node_key"] = (
        "OVERLAP||"
        + df["overlap_category"].astype(str)
    )


    # --------------------------------------------------------
    # 3. Build LOT1 nodes
    # --------------------------------------------------------

    lot1_nodes = (
        df[
            [
                "lot1_node_key",
                "lot1_class",
                "lot1_sort"
            ]
        ]
        .drop_duplicates()
        .sort_values("lot1_sort")
    )


    # --------------------------------------------------------
    # 4. Build LOT2 nodes
    # --------------------------------------------------------

    lot2_nodes = (
        df[
            [
                "lot2_node_key",
                "lot1_class",
                "lot2_class",
                "lot1_sort",
                "lot2_sort"
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "lot1_sort",
                "lot2_sort"
            ]
        )
    )


    # --------------------------------------------------------
    # 5. Build overlap nodes
    # --------------------------------------------------------

    overlap_nodes = (
        df[
            [
                "overlap_node_key",
                "overlap_category",
                "overlap_sort"
            ]
        ]
        .drop_duplicates()
        .sort_values("overlap_sort")
    )


    # --------------------------------------------------------
    # 6. Combine all node keys and labels
    # --------------------------------------------------------

    node_keys = (
        lot1_nodes["lot1_node_key"].tolist()
        + lot2_nodes["lot2_node_key"].tolist()
        + overlap_nodes["overlap_node_key"].tolist()
    )

    node_labels = (
        lot1_nodes["lot1_class"].tolist()
        + lot2_nodes["lot2_class"].tolist()
        + overlap_nodes["overlap_category"].tolist()
    )

    node_index = {
        key: i
        for i, key in enumerate(node_keys)
    }


    # --------------------------------------------------------
    # 7. Fix horizontal positions
    #
    # LOT1 at left
    # LOT2 in middle
    # overlap category at right
    # --------------------------------------------------------

    lot1_x = [0.01] * len(lot1_nodes)
    lot2_x = [0.50] * len(lot2_nodes)
    overlap_x = [0.90] * len(overlap_nodes)

    node_x = lot1_x + lot2_x + overlap_x


    def evenly_spaced_y(n, top=0.03, bottom=0.92):
        if n <= 1:
            return [0.5]

        return [
            top + (bottom - top) * i / (n - 1)
            for i in range(n)
        ]


    node_y = (
        evenly_spaced_y(len(lot1_nodes))
        + evenly_spaced_y(len(lot2_nodes))
        + evenly_spaced_y(len(overlap_nodes))
    )


    # --------------------------------------------------------
    # 8. LOT1 -> LOT2 links
    # --------------------------------------------------------

    lot1_lot2_links = (
        df
        .groupby(
            [
                "lot1_node_key",
                "lot2_node_key",
                "lot1_class",
                "lot2_class"
            ],
            as_index=False
        )
        .agg(
            patient_n=("patient_n", "sum")
        )
    )

    lot1_lot2_links["pct"] = (
        lot1_lot2_links["patient_n"]
        / denominator_n
        * 100
    )


    # --------------------------------------------------------
    # 9. LOT2 -> overlap-category links
    # --------------------------------------------------------

    lot2_overlap_links = (
        df
        .groupby(
            [
                "lot2_node_key",
                "overlap_node_key",
                "lot1_class",
                "lot2_class",
                "overlap_category"
            ],
            as_index=False
        )
        .agg(
            patient_n=("patient_n", "sum")
        )
    )

    lot2_overlap_links["pct"] = (
        lot2_overlap_links["patient_n"]
        / denominator_n
        * 100
    )


    # --------------------------------------------------------
    # 10. Combine links
    # --------------------------------------------------------

    source = []
    target = []
    value = []
    customdata = []


    # LOT1 -> LOT2
    for _, row in lot1_lot2_links.iterrows():

        source.append(
            node_index[row["lot1_node_key"]]
        )

        target.append(
            node_index[row["lot2_node_key"]]
        )

        value.append(
            int(row["patient_n"])
        )

        customdata.append(
            (
                f'{row["lot1_class"]} → '
                f'{row["lot2_class"]}'
                f'<br>Patients: {int(row["patient_n"]):,}'
                f'<br>Percent: {row["pct"]:.2f}%'
            )
        )


    # LOT2 -> overlap
    for _, row in lot2_overlap_links.iterrows():

        source.append(
            node_index[row["lot2_node_key"]]
        )

        target.append(
            node_index[row["overlap_node_key"]]
        )

        value.append(
            int(row["patient_n"])
        )

        customdata.append(
            (
                f'{row["lot1_class"]} → '
                f'{row["lot2_class"]}'
                f'<br>{row["overlap_category"]}'
                f'<br>Patients: {int(row["patient_n"]):,}'
                f'<br>Percent: {row["pct"]:.2f}%'
            )
        )


    # --------------------------------------------------------
    # 11. Create plot
    # --------------------------------------------------------

    fig = go.Figure(
        data=[
            go.Sankey(
                arrangement="fixed",

                node=dict(
                    pad=12,
                    thickness=18,
                    line=dict(
                        width=0.5
                    ),
                    label=node_labels,
                    x=node_x,
                    y=node_y,
                    hovertemplate=(
                        "%{label}"
                        "<extra></extra>"
                    )
                ),

                link=dict(
                    source=source,
                    target=target,
                    value=value,
                    customdata=customdata,
                    hovertemplate=(
                        "%{customdata}"
                        "<extra></extra>"
                    )
                )
            )
        ]
    )


    plot_height = max(
        1050,
        len(lot2_nodes) * 30
    )

    fig.update_layout(
        title=dict(
            text=(
                f"{plot_title}"
                f"<br><sup>N = {denominator_n:,}</sup>"
            ),
            x=0.5
        ),
        font=dict(
            size=11
        ),
        width=1800,
        height=plot_height,
        margin=dict(
            l=80,
            r=260,
            t=100,
            b=330
        )
    )


    # --------------------------------------------------------
    # 11A. Add explanatory footnote
    # --------------------------------------------------------

    if footnote_text is not None:

        fig.add_annotation(
            text=(
                "<span style='font-size:12px'>"
                + footnote_text
                + "</span>"
            ),
            xref="paper",
            yref="paper",
            x=0,
            y=-0.12,
            xanchor="left",
            yanchor="top",
            showarrow=False,
            align="left",
            width=1600
        )


    # Show plot in Notebook
    fig.show()


    # --------------------------------------------------------
    # 12. Export files
    # --------------------------------------------------------

    html_path = os.path.join(
        output_dir,
        file_name + ".html"
    )

    png_path = os.path.join(
        output_dir,
        file_name + ".png"
    )


    # HTML always works and remains interactive
    fig.write_html(
        html_path
    )


    # PNG requires kaleido
    try:
        fig.write_image(
            png_path,
            width=1800,
            height=plot_height,
            scale=2
        )

        print("PNG exported:")
        print(png_path)

    except Exception as error:

        print("PNG was not exported.")
        print("HTML was exported successfully.")
        print("To export PNG, install kaleido if permitted.")
        print("Error:", error)


    print("HTML exported:")
    print(html_path)

    print("-" * 60)


    return fig

In [ ]:
# Environment setup note: install package only if needed in a fresh environment.
# !pip install --user kaleido==0.2.1

In [ ]:
fig_lot_1y = create_lot_sankey(
    summary_df=lot_summary_1y,
    plot_title=(
        "Antidepressant LOT1 to LOT2 treatment pattern, "
        "1-year follow-up"
    ),
    file_name="Table6_Sankey_LOT_1year",
    footnote_text=make_lot_sankey_footnote(
        "1-year follow-up window"
    )
)

In [ ]:
fig_lot_2y = create_lot_sankey(
    summary_df=lot_summary_2y,
    plot_title=(
        "Antidepressant LOT1 to LOT2 treatment pattern, "
        "2-year follow-up"
    ),
    file_name="Table6_Sankey_LOT_2year",
    footnote_text=make_lot_sankey_footnote(
        "2-year follow-up window"
    )
)

In [ ]:
fig_lot_entire = create_lot_sankey(
    summary_df=lot_summary_entire,
    plot_title=(
        "Antidepressant LOT1 to LOT2 treatment pattern, "
        "entire follow-up"
    ),
    file_name="Table6_Sankey_LOT_entire_FU",
    footnote_text=make_lot_sankey_footnote(
        "entire available follow-up period"
    )
)

In [ ]:
fig_lot_recent5y = create_lot_sankey(
    summary_df=lot_summary_recent5y,
    plot_title=(
        "Antidepressant LOT1 to LOT2 treatment pattern, "
        "recent 5-year window"
    ),
    file_name="Table6_Sankey_LOT_recent5year",
    footnote_text=make_lot_sankey_footnote(
        "recent 5-year observation window"
    )
)

## Table 6 five-category overlap summary

This section creates detailed overlap categories for LOT transition summaries and Sankey plots.


In [ ]:
# ============================================================
# TABLE 6 - FIVE-CATEGORY LOT SUMMARY
# LOT1 -> LOT2 -> detailed overlap category
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# Fixed display order
# ------------------------------------------------------------

lot1_class_order = [
    "SSRI",
    "Atypical Antidepressants",
    "SNRI",
    "TCA",
    "MAOI"
]

lot1_rank = {
    class_name: rank
    for rank, class_name in enumerate(lot1_class_order)
}


# LOT2 order matching teacher's display
lot2_class_order = [
    "Atypical Antidepressants",
    "MAOI",
    "SNRI",
    "SSRI",
    "TCA"
]

lot2_rank = {
    class_name: rank
    for rank, class_name in enumerate(lot2_class_order)
}


# Five-category overlap order
overlap_5cat_order = [
    "No overlap (0 days)",
    "Short overlap (1-29 days)",
    "Moderate overlap (30-89 days)",
    "Long overlap (>=90 days)",
    "No qualifying non-index class in window"
]

overlap_5cat_rank = {
    category: rank
    for rank, category in enumerate(overlap_5cat_order)
}


def make_ordered_lot2_combo_5cat(class_series):
    """
    Combine same-day LOT2 classes using fixed class order.
    """

    unique_classes = list(
        set(class_series.dropna().astype(str))
    )

    unique_classes = sorted(
        unique_classes,
        key=lambda x: lot2_rank.get(x, 999)
    )

    return " + ".join(unique_classes)


def make_lot2_sort_key_5cat(lot2_label):
    """
    Sort single and combined LOT2 labels.
    """

    if pd.isna(lot2_label):
        return (999, 999, (999,))

    components = [
        component.strip()
        for component in str(lot2_label).split(" + ")
    ]

    component_ranks = tuple(
        sorted(
            lot2_rank.get(component, 999)
            for component in components
        )
    )

    return (
        component_ranks[0],
        len(component_ranks),
        component_ranks
    )


def build_lot_tables_5cat(
    class_episodes_df,
    lot1_episodes_df,
    window_name,
    followup_days=None,
    recent_years=None,
    data_cutoff=None
):
    """
    Build LOT1 -> earliest LOT2 table using five overlap categories.

    Outputs:
    1. patient_detail_5cat
    2. lot_summary_5cat
    3. lot1_check_5cat
    """

    if data_cutoff is None:
        raise ValueError("data_cutoff must be provided.")


    # --------------------------------------------------------
    # 1. Eligible patients
    # --------------------------------------------------------

    base_patients = lot1_episodes_df[
        [
            "person_id",
            "index_drug_date",
            "lot1_class",
            "lot1_start",
            "lot1_end"
        ]
    ].copy()


    # Recent 5-year cohort:
    # restrict patients by index date
    if recent_years is not None:

        recent_start_date = (
            data_cutoff
            - pd.DateOffset(years=recent_years)
        )

        base_patients = base_patients[
            (
                base_patients["index_drug_date"]
                >= recent_start_date
            )
            & (
                base_patients["index_drug_date"]
                <= data_cutoff
            )
        ].copy()


    denominator_n = (
        base_patients["person_id"].nunique()
    )


    # --------------------------------------------------------
    # 2. Attach all class-level episodes
    # --------------------------------------------------------

    lot2_candidates = class_episodes_df.merge(
        base_patients,
        on="person_id",
        how="inner"
    )


    # Qualifying LOT2:
    # post-index and different from LOT1 class
    lot2_candidates = lot2_candidates[
        (
            lot2_candidates["episode_start"]
            > lot2_candidates["index_drug_date"]
        )
        & (
            lot2_candidates["ad_class"]
            != lot2_candidates["lot1_class"]
        )
        & (
            lot2_candidates["episode_start"]
            <= data_cutoff
        )
    ].copy()


    # --------------------------------------------------------
    # 3. Apply 1-year / 2-year window
    # --------------------------------------------------------

    if followup_days is not None:

        lot2_candidates["followup_window_end"] = (
            lot2_candidates["index_drug_date"]
            + pd.to_timedelta(
                followup_days,
                unit="D"
            )
        )

        lot2_candidates = lot2_candidates[
            (
                lot2_candidates["episode_start"]
                <= lot2_candidates["followup_window_end"]
            )
        ].copy()


    # --------------------------------------------------------
    # 4. Calculate inclusive overlap for candidate episodes
    # --------------------------------------------------------

    lot2_candidates["overlap_start"] = (
        lot2_candidates[
            [
                "lot1_start",
                "episode_start"
            ]
        ]
        .max(axis=1)
    )

    lot2_candidates["overlap_end"] = (
        lot2_candidates[
            [
                "lot1_end",
                "episode_end"
            ]
        ]
        .min(axis=1)
    )


    # Inclusive overlap:
    # overlap_end - overlap_start + 1
    lot2_candidates["overlap_days"] = (
        (
            lot2_candidates["overlap_end"]
            - lot2_candidates["overlap_start"]
        ).dt.days
        + 1
    )

    lot2_candidates["overlap_days"] = (
        lot2_candidates["overlap_days"]
        .clip(lower=0)
        .astype(int)
    )


    # --------------------------------------------------------
    # 5. Find earliest LOT2 date per patient
    # --------------------------------------------------------

    earliest_lot2_date = (
        lot2_candidates
        .groupby(
            "person_id",
            as_index=False
        )
        .agg(
            lot2_start=("episode_start", "min")
        )
    )


    earliest_lot2_records = lot2_candidates.merge(
        earliest_lot2_date,
        left_on=[
            "person_id",
            "episode_start"
        ],
        right_on=[
            "person_id",
            "lot2_start"
        ],
        how="inner"
    )


    # --------------------------------------------------------
    # 6. Collapse same-day LOT2 classes
    # --------------------------------------------------------

    patient_lot2 = (
        earliest_lot2_records
        .groupby(
            "person_id",
            as_index=False
        )
        .agg(
            lot2_start=(
                "lot2_start",
                "first"
            ),

            lot2_class=(
                "ad_class",
                make_ordered_lot2_combo_5cat
            ),

            lot2_class_n=(
                "ad_class",
                lambda x: x.dropna().nunique()
            ),

            # Footnote:
            # maximum overlap across same-day LOT2 classes
            overlap_days=(
                "overlap_days",
                "max"
            ),

            lot2_episode_end=(
                "episode_end",
                "max"
            )
        )
    )


    # --------------------------------------------------------
    # 7. Merge LOT2 back to all eligible patients
    # --------------------------------------------------------

    patient_detail = base_patients.merge(
        patient_lot2,
        on="person_id",
        how="left"
    )


    no_qualifying_lot2 = (
        patient_detail["lot2_class"].isna()
    )


    # For no qualifying patients, show LOT1 again under LOT2
    patient_detail.loc[
        no_qualifying_lot2,
        "lot2_class"
    ] = patient_detail.loc[
        no_qualifying_lot2,
        "lot1_class"
    ]


    # --------------------------------------------------------
    # 8. Assign five overlap categories
    # --------------------------------------------------------

    patient_detail["overlap_category"] = np.select(
        [
            no_qualifying_lot2,

            patient_detail["overlap_days"] == 0,

            patient_detail["overlap_days"].between(
                1,
                29
            ),

            patient_detail["overlap_days"].between(
                30,
                89
            ),

            patient_detail["overlap_days"] >= 90
        ],
        [
            "No qualifying non-index class in window",

            "No overlap (0 days)",

            "Short overlap (1-29 days)",

            "Moderate overlap (30-89 days)",

            "Long overlap (>=90 days)"
        ],
        default="Check"
    )


    patient_detail["analysis_window"] = window_name


    # --------------------------------------------------------
    # 9. Patient-level detail table
    # --------------------------------------------------------

    patient_detail = patient_detail[
        [
            "person_id",
            "analysis_window",
            "index_drug_date",
            "lot1_class",
            "lot1_start",
            "lot1_end",
            "lot2_class",
            "lot2_class_n",
            "lot2_start",
            "lot2_episode_end",
            "overlap_days",
            "overlap_category"
        ]
    ].copy()


    patient_detail["lot1_order"] = (
        patient_detail["lot1_class"]
        .map(lot1_rank)
    )

    patient_detail["lot2_order"] = (
        patient_detail["lot2_class"]
        .apply(make_lot2_sort_key_5cat)
    )

    patient_detail["overlap_order"] = (
        patient_detail["overlap_category"]
        .map(overlap_5cat_rank)
    )


    patient_detail = (
        patient_detail
        .sort_values(
            [
                "lot1_order",
                "lot2_order",
                "overlap_order",
                "person_id"
            ]
        )
        .drop(
            columns=[
                "lot1_order",
                "lot2_order",
                "overlap_order"
            ]
        )
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # 10. Complete summary table
    # --------------------------------------------------------

    lot_summary = (
        patient_detail
        .groupby(
            [
                "lot1_class",
                "lot2_class",
                "overlap_category"
            ],
            as_index=False
        )
        .agg(
            patient_n=("person_id", "nunique")
        )
    )


    lot_summary["pct"] = (
        lot_summary["patient_n"]
        / denominator_n
    )


    lot_summary["lot1_order"] = (
        lot_summary["lot1_class"]
        .map(lot1_rank)
    )

    lot_summary["lot2_order"] = (
        lot_summary["lot2_class"]
        .apply(make_lot2_sort_key_5cat)
    )

    lot_summary["overlap_order"] = (
        lot_summary["overlap_category"]
        .map(overlap_5cat_rank)
    )


    lot_summary = (
        lot_summary
        .sort_values(
            [
                "lot1_order",
                "lot2_order",
                "overlap_order"
            ]
        )
        .drop(
            columns=[
                "lot1_order",
                "lot2_order",
                "overlap_order"
            ]
        )
        .reset_index(drop=True)
    )


    lot_summary.insert(
        0,
        "analysis_window",
        window_name
    )


    lot_summary = lot_summary[
        [
            "analysis_window",
            "lot1_class",
            "lot2_class",
            "overlap_category",
            "patient_n",
            "pct"
        ]
    ]


    # --------------------------------------------------------
    # 11. LOT1 count check
    # --------------------------------------------------------

    lot1_check = (
        patient_detail
        .groupby(
            "lot1_class",
            as_index=False
        )
        .agg(
            patient_n=("person_id", "nunique")
        )
    )


    lot1_check["lot1_order"] = (
        lot1_check["lot1_class"]
        .map(lot1_rank)
    )


    lot1_check = (
        lot1_check
        .sort_values("lot1_order")
        .drop(columns="lot1_order")
        .reset_index(drop=True)
    )


    lot1_check["pct"] = (
        lot1_check["patient_n"]
        / denominator_n
    )


    total_row = pd.DataFrame({
        "lot1_class": ["Total"],
        "patient_n": [
            lot1_check["patient_n"].sum()
        ],
        "pct": [1.0]
    })


    lot1_check = pd.concat(
        [
            lot1_check,
            total_row
        ],
        ignore_index=True
    )


    # --------------------------------------------------------
    # 12. Print validation
    # --------------------------------------------------------

    print("=" * 60)
    print("Window:", window_name)
    print("Eligible patients:", denominator_n)
    print(
        "Patient-detail rows:",
        len(patient_detail)
    )
    print(
        "Summary patient total:",
        lot_summary["patient_n"].sum()
    )
    print("=" * 60)


    return (
        patient_detail,
        lot_summary,
        lot1_check
    )

In [ ]:
# ============================================================
# BUILD FOUR FIVE-CATEGORY LOT SUMMARIES
# ============================================================

patient_lot_1y_5cat, lot_summary_1y_5cat, lot1_check_1y_5cat = (
    build_lot_tables_5cat(
        class_episodes_df=class_episodes,
        lot1_episodes_df=lot1_episodes,
        window_name="1-year FU",
        followup_days=365,
        recent_years=None,
        data_cutoff=data_cutoff
    )
)


patient_lot_2y_5cat, lot_summary_2y_5cat, lot1_check_2y_5cat = (
    build_lot_tables_5cat(
        class_episodes_df=class_episodes,
        lot1_episodes_df=lot1_episodes,
        window_name="2-year FU",
        followup_days=730,
        recent_years=None,
        data_cutoff=data_cutoff
    )
)


patient_lot_entire_5cat, lot_summary_entire_5cat, lot1_check_entire_5cat = (
    build_lot_tables_5cat(
        class_episodes_df=class_episodes,
        lot1_episodes_df=lot1_episodes,
        window_name="Entire FU",
        followup_days=None,
        recent_years=None,
        data_cutoff=data_cutoff
    )
)



patient_lot_recent5y_5cat, lot_summary_recent5y_5cat, lot1_check_recent5y_5cat = (
    build_lot_tables_5cat(
        class_episodes_df=class_episodes,
        lot1_episodes_df=lot1_episodes,
        window_name="Recent 5-year cohort",
        followup_days=None,
        recent_years=5,
        data_cutoff=data_cutoff
    )
)

In [ ]:
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None
):
    display(lot_summary_1y_5cat)

In [ ]:
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None
):
    display(lot_summary_2y_5cat)

In [ ]:
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None
):
    display(lot_summary_entire_5cat)

In [ ]:
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None
):
    display(lot_summary_recent5y_5cat)

In [ ]:
# ============================================================
# FIVE-CATEGORY SANKEY PLOT FUNCTION
# ============================================================

import os
import plotly.graph_objects as go


output_dir = "/home/jupyter/workspaces/mddprojectcontrolledtier"

os.makedirs(
    output_dir,
    exist_ok=True
)


def make_lot_sankey_footnote_5cat(window_label):
    """
    Create explanatory footnote for five-category LOT Sankey plots.
    Line breaks are added for better readability in HTML/PNG outputs.
    """

    return (
        "<b>Note.</b> LOT = line of therapy.<br>"
        "LOT1 represents the first observed antidepressant treatment class after the index MDD diagnosis. "
        f"LOT2 represents the first qualifying subsequent non-index antidepressant class identified within the {window_label}.<br>"
        "Sankey flow width represents patient count, not prescription count or drug exposure count. "
        "Percentages shown in hover labels are calculated using the total N for the current plot.<br>"
        "Detailed overlap categories describe overlapping exposure between LOT1 and LOT2: "
        "no overlap, short overlap &lt;30 days, moderate overlap 30–89 days, and long overlap ≥90 days.<br>"
        "The no qualifying non-index class category indicates that no subsequent antidepressant class meeting LOT2 criteria was identified within the specified window.<br>"
        "Antidepressant classes include SSRI, SNRI, atypical antidepressants, TCA, and MAOI."
    )



def create_lot_sankey_5cat(
    summary_df,
    plot_title,
    file_name,
    footnote_text=None
):
    """
    Create a three-level Sankey plot:

    LOT1 -> LOT2 -> detailed overlap category
    """

    df = summary_df.copy()

    df["patient_n"] = pd.to_numeric(
        df["patient_n"],
        errors="coerce"
    ).fillna(0).astype(int)

    denominator_n = int(
        df["patient_n"].sum()
    )

    df["plot_pct"] = (
        df["patient_n"]
        / denominator_n
        * 100
    )


    # --------------------------------------------------------
    # Sort rows
    # --------------------------------------------------------

    df["lot1_sort"] = (
        df["lot1_class"]
        .map(lot1_rank)
        .fillna(999)
    )

    df["lot2_sort"] = (
        df["lot2_class"]
        .apply(make_lot2_sort_key_5cat)
    )

    df["overlap_sort"] = (
        df["overlap_category"]
        .map(overlap_5cat_rank)
        .fillna(999)
    )

    df = (
        df
        .sort_values(
            [
                "lot1_sort",
                "lot2_sort",
                "overlap_sort"
            ]
        )
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # Create unique node keys
    # LOT2 is unique within LOT1 block
    # --------------------------------------------------------

    df["lot1_node_key"] = (
        "LOT1||"
        + df["lot1_class"].astype(str)
    )

    df["lot2_node_key"] = (
        "LOT2||"
        + df["lot1_class"].astype(str)
        + "||"
        + df["lot2_class"].astype(str)
    )

    df["overlap_node_key"] = (
        "OVERLAP||"
        + df["overlap_category"].astype(str)
    )


    lot1_nodes = (
        df[
            [
                "lot1_node_key",
                "lot1_class",
                "lot1_sort"
            ]
        ]
        .drop_duplicates()
        .sort_values("lot1_sort")
    )


    lot2_nodes = (
        df[
            [
                "lot2_node_key",
                "lot1_class",
                "lot2_class",
                "lot1_sort",
                "lot2_sort"
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "lot1_sort",
                "lot2_sort"
            ]
        )
    )


    overlap_nodes = (
        df[
            [
                "overlap_node_key",
                "overlap_category",
                "overlap_sort"
            ]
        ]
        .drop_duplicates()
        .sort_values("overlap_sort")
    )


    node_keys = (
        lot1_nodes["lot1_node_key"].tolist()
        + lot2_nodes["lot2_node_key"].tolist()
        + overlap_nodes["overlap_node_key"].tolist()
    )

    node_labels = (
        lot1_nodes["lot1_class"].tolist()
        + lot2_nodes["lot2_class"].tolist()
        + overlap_nodes["overlap_category"].tolist()
    )


    node_index = {
        key: i
        for i, key in enumerate(node_keys)
    }


    # --------------------------------------------------------
    # Fixed node positions
    # --------------------------------------------------------

    lot1_x = [0.01] * len(lot1_nodes)
    lot2_x = [0.45] * len(lot2_nodes)
    overlap_x = [0.90] * len(overlap_nodes)

    node_x = lot1_x + lot2_x + overlap_x


    def evenly_spaced_y(
        n,
        top=0.03,
        bottom=0.88
    ):
        if n <= 1:
            return [0.5]

        return [
            top + (bottom - top) * i / (n - 1)
            for i in range(n)
        ]


    node_y = (
        evenly_spaced_y(len(lot1_nodes))
        + evenly_spaced_y(len(lot2_nodes))
        + evenly_spaced_y(len(overlap_nodes))
    )


    # --------------------------------------------------------
    # Links: LOT1 -> LOT2
    # --------------------------------------------------------

    lot1_lot2_links = (
        df
        .groupby(
            [
                "lot1_node_key",
                "lot2_node_key",
                "lot1_class",
                "lot2_class"
            ],
            as_index=False
        )
        .agg(
            patient_n=("patient_n", "sum")
        )
    )

    lot1_lot2_links["pct"] = (
        lot1_lot2_links["patient_n"]
        / denominator_n
        * 100
    )


    # --------------------------------------------------------
    # Links: LOT2 -> overlap category
    # --------------------------------------------------------

    lot2_overlap_links = (
        df
        .groupby(
            [
                "lot2_node_key",
                "overlap_node_key",
                "lot1_class",
                "lot2_class",
                "overlap_category"
            ],
            as_index=False
        )
        .agg(
            patient_n=("patient_n", "sum")
        )
    )

    lot2_overlap_links["pct"] = (
        lot2_overlap_links["patient_n"]
        / denominator_n
        * 100
    )


    source = []
    target = []
    value = []
    customdata = []


    for _, row in lot1_lot2_links.iterrows():

        source.append(
            node_index[row["lot1_node_key"]]
        )

        target.append(
            node_index[row["lot2_node_key"]]
        )

        value.append(
            int(row["patient_n"])
        )

        customdata.append(
            (
                f'{row["lot1_class"]} → '
                f'{row["lot2_class"]}'
                f'<br>Patients: {int(row["patient_n"]):,}'
                f'<br>Percent: {row["pct"]:.2f}%'
            )
        )


    for _, row in lot2_overlap_links.iterrows():

        source.append(
            node_index[row["lot2_node_key"]]
        )

        target.append(
            node_index[row["overlap_node_key"]]
        )

        value.append(
            int(row["patient_n"])
        )

        customdata.append(
            (
                f'{row["lot1_class"]} → '
                f'{row["lot2_class"]}'
                f'<br>{row["overlap_category"]}'
                f'<br>Patients: {int(row["patient_n"]):,}'
                f'<br>Percent: {row["pct"]:.2f}%'
            )
        )


    fig_height = max(
        1150,
        len(lot2_nodes) * 32
    )


    fig = go.Figure(
        data=[
            go.Sankey(
                arrangement="fixed",

                node=dict(
                    pad=12,
                    thickness=18,
                    line=dict(
                        width=0.5
                    ),
                    label=node_labels,
                    x=node_x,
                    y=node_y,
                    hovertemplate=(
                        "%{label}"
                        "<extra></extra>"
                    )
                ),

                link=dict(
                    source=source,
                    target=target,
                    value=value,
                    customdata=customdata,
                    hovertemplate=(
                        "%{customdata}"
                        "<extra></extra>"
                    )
                )
            )
        ]
    )


    fig.update_layout(
        title=dict(
            text=(
                f"{plot_title}"
                f"<br><sup>N = {denominator_n:,}</sup>"
            ),
            x=0.5
        ),
        font=dict(
            size=11
        ),
        width=1900,
        height=fig_height,
        margin=dict(
            l=80,
            r=260,
            t=100,
            b=360
        )
    )


    # --------------------------------------------------------
    # Add explanatory footnote
    # --------------------------------------------------------

    if footnote_text is not None:

        fig.add_annotation(
            text=(
                "<span style='font-size:12px'>"
                + footnote_text
                + "</span>"
            ),
            xref="paper",
            yref="paper",
            x=0,
            y=-0.13,
            xanchor="left",
            yanchor="top",
            showarrow=False,
            align="left",
            width=1700
        )

    fig.show()


    html_path = os.path.join(
        output_dir,
        file_name + ".html"
    )

    png_path = os.path.join(
        output_dir,
        file_name + ".png"
    )


    fig.write_html(
        html_path
    )


    try:
        fig.write_image(
            png_path,
            width=1900,
            height=fig_height,
            scale=2
        )

        print("PNG exported:")
        print(png_path)

    except Exception as error:

        print("PNG was not exported.")
        print("HTML was exported successfully.")
        print("If needed, install kaleido:")
        print("!pip install --user kaleido==0.2.1")
        print("Error:", error)


    print("HTML exported:")
    print(html_path)
    print("-" * 60)


    return fig

In [ ]:
fig_lot_1y_5cat = create_lot_sankey_5cat(
    summary_df=lot_summary_1y_5cat,
    plot_title=(
        "Antidepressant LOT1 to LOT2 treatment pattern, "
        "1-year follow-up, detailed overlap categories"
    ),
    file_name="Table6_Sankey_LOT_1year_5cat",
    footnote_text=make_lot_sankey_footnote_5cat(
        "1-year follow-up window"
    )
)

In [ ]:
fig_lot_2y_5cat = create_lot_sankey_5cat(
    summary_df=lot_summary_2y_5cat,
    plot_title=(
        "Antidepressant LOT1 to LOT2 treatment pattern, "
        "2-year follow-up, detailed overlap categories"
    ),
    file_name="Table6_Sankey_LOT_2year_5cat",
    footnote_text=make_lot_sankey_footnote_5cat(
        "2-year follow-up window"
    )
)

In [ ]:
fig_lot_entire_5cat = create_lot_sankey_5cat(
    summary_df=lot_summary_entire_5cat,
    plot_title=(
        "Antidepressant LOT1 to LOT2 treatment pattern, "
        "entire follow-up, detailed overlap categories"
    ),
    file_name="Table6_Sankey_LOT_entire_FU_5cat",
    footnote_text=make_lot_sankey_footnote_5cat(
        "entire available follow-up period"
    )
)

In [ ]:
fig_lot_recent5y_5cat = create_lot_sankey_5cat(
    summary_df=lot_summary_recent5y_5cat,
    plot_title=(
        "Antidepressant LOT1 to LOT2 treatment pattern, "
        "recent 5-year window, detailed overlap categories"
    ),
    file_name="Table6_Sankey_LOT_recent5year_5cat",
    footnote_text=make_lot_sankey_footnote_5cat(
        "recent 5-year observation window"
    )
)

In [ ]:
# ============================================================
# EXPORT FIVE-CATEGORY LOT SUMMARIES
# ============================================================

output_path_5cat = (
    "/home/jupyter/workspaces/"
    "mddprojectcontrolledtier/"
    "Table6_LOT_summary_5cat_results.xlsx"
)


with pd.ExcelWriter(
    output_path_5cat,
    engine="openpyxl"
) as writer:

    lot_summary_1y_5cat.to_excel(
        writer,
        sheet_name="lot_summary_1y_5cat",
        index=False
    )

    lot_summary_2y_5cat.to_excel(
        writer,
        sheet_name="lot_summary_2y_5cat",
        index=False
    )

    lot_summary_entire_5cat.to_excel(
        writer,
        sheet_name="lot_summary_entire_5cat",
        index=False
    )

    lot_summary_recent5y_5cat.to_excel(
        writer,
        sheet_name="lot_summary_recent5y_5cat",
        index=False
    )

    lot1_check_1y_5cat.to_excel(
        writer,
        sheet_name="lot1_check_1y_5cat",
        index=False
    )

    lot1_check_2y_5cat.to_excel(
        writer,
        sheet_name="lot1_check_2y_5cat",
        index=False
    )

    lot1_check_entire_5cat.to_excel(
        writer,
        sheet_name="lot1_check_entire_5cat",
        index=False
    )

    lot1_check_recent5y_5cat.to_excel(
        writer,
        sheet_name="lot1_check_recent5y_5cat",
        index=False
    )


print("Exported to:")
print(output_path_5cat)

## Table 7: Single-class LOT2 overlap summary

This section summarizes overlap days for single-class LOT2 transitions in the recent 5-year cohort.


In [ ]:
# ============================================================
# TABLE 7 - SINGLE-CLASS LOT2 OVERLAP SUMMARY
# Recent 5-year index cohort
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Set denominator
# ------------------------------------------------------------

recent5y_denominator_n = (
    patient_lot_recent5y["person_id"]
    .nunique()
)

print(
    "Recent 5-year denominator:",
    recent5y_denominator_n
)


# ------------------------------------------------------------
# 2. Keep only patients with qualifying LOT2
#    and LOT2 as a single class
# ------------------------------------------------------------

table7_single_lot2 = (
    patient_lot_recent5y[
        (
            patient_lot_recent5y["lot2_class"].notna()
        )
        & (
            patient_lot_recent5y["overlap_category"]
            != "No qualifying non-index class in window"
        )
        & (
            patient_lot_recent5y["lot2_class_n"] == 1
        )
    ]
    .copy()
)


# Make sure overlap_days is numeric
table7_single_lot2["overlap_days"] = pd.to_numeric(
    table7_single_lot2["overlap_days"],
    errors="coerce"
).fillna(0).astype(int)


print(
    "Patients with single-class LOT2 transitions:",
    table7_single_lot2["person_id"].nunique()
)

display(
    table7_single_lot2.head()
)

In [ ]:
# ============================================================
# Helper function for overlap-days summary
# ============================================================

def summarize_overlap_days(group):
    """
    Summarize overlap_days for one LOT1 -> LOT2 transition.
    Includes patients with overlap_days = 0.
    """

    overlap = group["overlap_days"].dropna()

    return pd.Series({
        "patient_n": group["person_id"].nunique(),

        "overlap_0_n": (
            group.loc[
                group["overlap_days"] == 0,
                "person_id"
            ]
            .nunique()
        ),

        "overlap_gt0_n": (
            group.loc[
                group["overlap_days"] > 0,
                "person_id"
            ]
            .nunique()
        ),

        "mean_overlap_days": overlap.mean(),

        "median_overlap_days": overlap.median(),

        "q1_overlap_days": overlap.quantile(0.25),

        "q3_overlap_days": overlap.quantile(0.75),

        "p95_overlap_days": overlap.quantile(0.95),

        "p99_overlap_days": overlap.quantile(0.99),

        "min_overlap_days": overlap.min(),

        "max_overlap_days": overlap.max()
    })

In [ ]:
# ============================================================
# Generate Table 7 summary
# ============================================================

table7_single_lot2_summary = (
    table7_single_lot2
    .groupby(
        [
            "lot1_class",
            "lot2_class"
        ]
    )
    .apply(
        summarize_overlap_days
    )
    .reset_index()
)


# Percent denominator is the recent 5-year index cohort
table7_single_lot2_summary["pct"] = (
    table7_single_lot2_summary["patient_n"]
    / recent5y_denominator_n
)


# ------------------------------------------------------------
# Add display order
# ------------------------------------------------------------

table7_single_lot2_summary["lot1_order"] = (
    table7_single_lot2_summary["lot1_class"]
    .map(lot1_rank)
)

table7_single_lot2_summary["lot2_order"] = (
    table7_single_lot2_summary["lot2_class"]
    .map(lot2_rank)
)


table7_single_lot2_summary = (
    table7_single_lot2_summary
    .sort_values(
        [
            "lot1_order",
            "lot2_order"
        ]
    )
    .drop(
        columns=[
            "lot1_order",
            "lot2_order"
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Round numeric columns for display
# ------------------------------------------------------------

round_cols = [
    "mean_overlap_days",
    "median_overlap_days",
    "q1_overlap_days",
    "q3_overlap_days",
    "p95_overlap_days",
    "p99_overlap_days",
    "min_overlap_days",
    "max_overlap_days"
]

table7_single_lot2_summary[round_cols] = (
    table7_single_lot2_summary[round_cols]
    .round(1)
)


# Optional: percent display column
table7_single_lot2_summary["pct_display"] = (
    table7_single_lot2_summary["pct"] * 100
).round(2)


# Reorder columns
table7_single_lot2_summary = table7_single_lot2_summary[
    [
        "lot1_class",
        "lot2_class",
        "patient_n",
        "pct",
        "pct_display",
        "overlap_0_n",
        "overlap_gt0_n",
        "mean_overlap_days",
        "median_overlap_days",
        "q1_overlap_days",
        "q3_overlap_days",
        "p95_overlap_days",
        "p99_overlap_days",
        "min_overlap_days",
        "max_overlap_days"
    ]
]


with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None
):
    display(table7_single_lot2_summary)

In [ ]:
# ============================================================
# Validation checks for Table 7
# ============================================================

print(
    "Recent 5-year denominator:",
    recent5y_denominator_n
)

print(
    "Table 7 single-class transition patients:",
    table7_single_lot2["person_id"].nunique()
)

print(
    "Sum of patient_n in Table 7:",
    table7_single_lot2_summary["patient_n"].sum()
)

print(
    "Check overlap_0_n + overlap_gt0_n equals patient_n:"
)

table7_check = table7_single_lot2_summary.copy()

table7_check["overlap_total_check"] = (
    table7_check["overlap_0_n"]
    + table7_check["overlap_gt0_n"]
)

table7_check["check_pass"] = (
    table7_check["overlap_total_check"]
    == table7_check["patient_n"]
)

display(
    table7_check[
        [
            "lot1_class",
            "lot2_class",
            "patient_n",
            "overlap_0_n",
            "overlap_gt0_n",
            "overlap_total_check",
            "check_pass"
        ]
    ]
)

In [ ]:
# ============================================================
# TABLE 7 - REORDER TO MATCH TEACHER'S EXCEL TABLE
# ============================================================

table7_row_order = [
    ("SSRI", "Atypical Antidepressants"),
    ("SSRI", "SNRI"),
    ("SSRI", "TCA"),

    ("Atypical Antidepressants", "SSRI"),
    ("Atypical Antidepressants", "SNRI"),
    ("Atypical Antidepressants", "TCA"),
    ("Atypical Antidepressants", "MAOI"),

    ("SNRI", "SSRI"),
    ("SNRI", "Atypical Antidepressants"),
    ("SNRI", "TCA"),

    ("TCA", "SSRI"),
    ("TCA", "Atypical Antidepressants"),
    ("TCA", "SNRI"),
    ("TCA", "MAOI"),

    ("MAOI", "Atypical Antidepressants"),
    ("MAOI", "SNRI")
]


table7_row_order_df = pd.DataFrame(
    table7_row_order,
    columns=[
        "lot1_class",
        "lot2_class"
    ]
)

table7_row_order_df["row_order"] = range(
    len(table7_row_order_df)
)


table7_single_lot2_summary_ordered = (
    table7_row_order_df
    .merge(
        table7_single_lot2_summary,
        on=[
            "lot1_class",
            "lot2_class"
        ],
        how="left"
    )
    .sort_values("row_order")
    .drop(columns="row_order")
    .reset_index(drop=True)
)
# ============================================================
# CLEAN DISPLAY TYPES
# ============================================================

integer_cols = [
    "patient_n",
    "overlap_0_n",
    "overlap_gt0_n",
    "min_overlap_days",
    "max_overlap_days"
]

for col in integer_cols:
    if col in table7_single_lot2_summary_ordered.columns:
        table7_single_lot2_summary_ordered[col] = (
            table7_single_lot2_summary_ordered[col]
            .fillna(0)
            .astype(int)
        )


decimal_cols = [
    "mean_overlap_days",
    "median_overlap_days",
    "q1_overlap_days",
    "q3_overlap_days",
    "p95_overlap_days",
    "p99_overlap_days"
]

for col in decimal_cols:
    if col in table7_single_lot2_summary_ordered.columns:
        table7_single_lot2_summary_ordered[col] = (
            table7_single_lot2_summary_ordered[col]
            .round(2)
        )
# ============================================================
# CREATE EXCEL-FRIENDLY TABLE 7 DISPLAY VERSION
# ============================================================

# Use current denominator from your dataframe
table7_denominator_n = recent5y_denominator_n

print(
    "Table 7 denominator:",
    table7_denominator_n
)


table7_single_lot2_excel = (
    table7_single_lot2_summary_ordered
    .copy()
)


table7_single_lot2_excel["patient_n_pct"] = (
    table7_single_lot2_excel["patient_n"].astype(str)
    + " ("
    + (
        table7_single_lot2_excel["patient_n"]
        / table7_denominator_n
        * 100
    ).round(1).astype(str)
    + "%)"
)


table7_single_lot2_excel["median_q1_q3"] = (
    table7_single_lot2_excel["median_overlap_days"]
    .round(2)
    .astype(str)
    + " ["
    + table7_single_lot2_excel["q1_overlap_days"]
    .round(2)
    .astype(str)
    + ", "
    + table7_single_lot2_excel["q3_overlap_days"]
    .round(2)
    .astype(str)
    + "]"
)


table7_single_lot2_excel["min_max"] = (
    table7_single_lot2_excel["min_overlap_days"]
    .astype(str)
    + ", "
    + table7_single_lot2_excel["max_overlap_days"]
    .astype(str)
)


table7_single_lot2_excel = table7_single_lot2_excel[
    [
        "lot1_class",
        "lot2_class",
        "patient_n",
        "patient_n_pct",
        "mean_overlap_days",
        "median_q1_q3",
        "p95_overlap_days",
        "p99_overlap_days",
        "min_max",
        "overlap_0_n",
        "overlap_gt0_n"
    ]
].copy()


table7_single_lot2_excel = table7_single_lot2_excel.rename(
    columns={
        "lot1_class": "LOT1",
        "lot2_class": "LOT2",
        "patient_n_pct": f"patient_n(%) among {table7_denominator_n}",
        "overlap_0_n": "overlap_0_n",
        "overlap_gt0_n": "overlap_gt0_n"
    }
)


with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None
):
    display(table7_single_lot2_excel)

In [ ]:
# ============================================================
# EXPORT TABLE 7 TO EXCEL
# ============================================================

table7_excel_path = os.path.join(
    output_dir,
    "Table7_single_class_LOT2_overlap_summary_recent5y.xlsx"
)


with pd.ExcelWriter(
    table7_excel_path,
    engine="openpyxl"
) as writer:

    table7_single_lot2_summary_ordered.to_excel(
        writer,
        sheet_name="Table7_raw_ordered",
        index=False
    )

    table7_single_lot2_excel.to_excel(
        writer,
        sheet_name="Table7_for_paste",
        index=False
    )


print(
    "Table 7 Excel exported:"
)

print(
    table7_excel_path
)

## Table 7A: SSRI-focused overlap summary

This section focuses on SSRI LOT1 patients and summarizes overlap with single LOT2 classes among patients with overlap_days > 0.


In [ ]:
# ============================================================
# TABLE 7A - SSRI LOT1 to single-class LOT2 only
# Statistics calculated among overlap_days > 0 only
# ============================================================

# ------------------------------------------------------------
# 1. Keep SSRI LOT1 and target single LOT2 classes
# ------------------------------------------------------------

ssri_lot2_order = [
    "Atypical Antidepressants",
    "SNRI",
    "TCA"
]


table7_ssri_single_lot2 = (
    table7_single_lot2[
        (
            table7_single_lot2["lot1_class"] == "SSRI"
        )
        & (
            table7_single_lot2["lot2_class"].isin(ssri_lot2_order)
        )
    ]
    .copy()
)


table7_ssri_single_lot2["overlap_days"] = pd.to_numeric(
    table7_ssri_single_lot2["overlap_days"],
    errors="coerce"
).fillna(0).astype(int)


# ------------------------------------------------------------
# 2. Helper function
#    patient_n_total includes everyone in this SSRI -> LOT2 group
#    summary stats use overlap_days > 0 only
# ------------------------------------------------------------

def summarize_ssri_lot2_overlap(group):
    """
    Summarize SSRI -> single LOT2 transition.

    patient_n_total includes all patients in the transition.
    overlap statistics are calculated among patients with overlap_days > 0 only.
    """

    patient_n_total = group["person_id"].nunique()

    overlap_0_n = (
        group.loc[
            group["overlap_days"] == 0,
            "person_id"
        ]
        .nunique()
    )

    overlap_gt0_n = (
        group.loc[
            group["overlap_days"] > 0,
            "person_id"
        ]
        .nunique()
    )

    overlap_gt0 = (
        group.loc[
            group["overlap_days"] > 0,
            "overlap_days"
        ]
        .dropna()
    )

    return pd.Series({
        "patient_n_total": patient_n_total,

        "patient_n_pct": (
            patient_n_total
            / recent5y_denominator_n
        ),

        "overlap_0_n": overlap_0_n,

        "overlap_0_pct": (
            overlap_0_n
            / patient_n_total
            if patient_n_total > 0
            else np.nan
        ),

        "overlap_gt0_n": overlap_gt0_n,

        "overlap_gt0_pct": (
            overlap_gt0_n
            / patient_n_total
            if patient_n_total > 0
            else np.nan
        ),

        "mean_overlap_days": overlap_gt0.mean(),

        "median_overlap_days": overlap_gt0.median(),

        "q1_overlap_days": overlap_gt0.quantile(0.25),

        "q3_overlap_days": overlap_gt0.quantile(0.75),

        "p95_overlap_days": overlap_gt0.quantile(0.95),

        "p99_overlap_days": overlap_gt0.quantile(0.99),

        "min_overlap_days": overlap_gt0.min(),

        "max_overlap_days": overlap_gt0.max()
    })


# ------------------------------------------------------------
# 3. Build SSRI summary table
# ------------------------------------------------------------

table7_ssri_single_lot2_summary = (
    table7_ssri_single_lot2
    .groupby(
        [
            "lot1_class",
            "lot2_class"
        ]
    )
    .apply(
        summarize_ssri_lot2_overlap
    )
    .reset_index()
)


# ------------------------------------------------------------
# 4. Force same order as teacher's Excel
# ------------------------------------------------------------

ssri_order_df = pd.DataFrame({
    "lot2_class": ssri_lot2_order,
    "lot2_order": range(len(ssri_lot2_order))
})


table7_ssri_single_lot2_summary = (
    table7_ssri_single_lot2_summary
    .merge(
        ssri_order_df,
        on="lot2_class",
        how="left"
    )
    .sort_values("lot2_order")
    .drop(columns="lot2_order")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. Clean display columns
# ------------------------------------------------------------

int_cols = [
    "patient_n_total",
    "overlap_0_n",
    "overlap_gt0_n",
    "min_overlap_days",
    "max_overlap_days"
]

for col in int_cols:
    table7_ssri_single_lot2_summary[col] = (
        table7_ssri_single_lot2_summary[col]
        .fillna(0)
        .astype(int)
    )


round_cols = [
    "mean_overlap_days",
    "median_overlap_days",
    "q1_overlap_days",
    "q3_overlap_days",
    "p95_overlap_days",
    "p99_overlap_days"
]

table7_ssri_single_lot2_summary[round_cols] = (
    table7_ssri_single_lot2_summary[round_cols]
    .round(2)
)


# ------------------------------------------------------------
# 6. Add display columns for percentages and median [Q1, Q3]
# ------------------------------------------------------------

table7_ssri_single_lot2_summary["patient_n_total_display"] = (
    table7_ssri_single_lot2_summary["patient_n_total"].astype(str)
    + " ("
    + (
        table7_ssri_single_lot2_summary["patient_n_pct"]
        * 100
    ).round(2).astype(str)
    + "%)"
)


table7_ssri_single_lot2_summary["overlap_0_display"] = (
    table7_ssri_single_lot2_summary["overlap_0_n"].astype(str)
    + " ("
    + (
        table7_ssri_single_lot2_summary["overlap_0_pct"]
        * 100
    ).round(2).astype(str)
    + "%)"
)


table7_ssri_single_lot2_summary["overlap_gt0_display"] = (
    table7_ssri_single_lot2_summary["overlap_gt0_n"].astype(str)
    + " ("
    + (
        table7_ssri_single_lot2_summary["overlap_gt0_pct"]
        * 100
    ).round(2).astype(str)
    + "%)"
)


table7_ssri_single_lot2_summary["median_q1_q3"] = (
    table7_ssri_single_lot2_summary["median_overlap_days"].astype(str)
    + " ["
    + table7_ssri_single_lot2_summary["q1_overlap_days"].astype(str)
    + ", "
    + table7_ssri_single_lot2_summary["q3_overlap_days"].astype(str)
    + "]"
)


table7_ssri_single_lot2_summary["min_max"] = (
    table7_ssri_single_lot2_summary["min_overlap_days"].astype(str)
    + ", "
    + table7_ssri_single_lot2_summary["max_overlap_days"].astype(str)
)


# ------------------------------------------------------------
# 7. Excel-friendly display table
# ------------------------------------------------------------

table7_ssri_single_lot2_display = (
    table7_ssri_single_lot2_summary[
        [
            "lot1_class",
            "lot2_class",
            "patient_n_total_display",
            "overlap_0_display",
            "overlap_gt0_display",
            "mean_overlap_days",
            "median_q1_q3",
            "p95_overlap_days",
            "p99_overlap_days",
            "min_max"
        ]
    ]
    .copy()
)


table7_ssri_single_lot2_display = (
    table7_ssri_single_lot2_display
    .rename(
        columns={
            "lot1_class": "LOT1",
            "lot2_class": "LOT2",
            "patient_n_total_display": "patient_n_total",
            "overlap_0_display": "overlap_0 没有overlap的人",
            "overlap_gt0_display": "overlap_gt0"
        }
    )
)


with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None
):
    display(table7_ssri_single_lot2_display)

In [ ]:
# ============================================================
# EXPORT SSRI TABLE 7A TO EXCEL
# ============================================================

table7_ssri_excel_path = os.path.join(
    output_dir,
    "Table7A_SSRI_single_LOT2_overlap_gt0_summary_recent5y.xlsx"
)


with pd.ExcelWriter(
    table7_ssri_excel_path,
    engine="openpyxl"
) as writer:

    table7_ssri_single_lot2_summary.to_excel(
        writer,
        sheet_name="SSRI_raw",
        index=False
    )

    table7_ssri_single_lot2_display.to_excel(
        writer,
        sheet_name="SSRI_for_paste",
        index=False
    )


print("SSRI Table 7A Excel exported:")
print(table7_ssri_excel_path)

## Table 7 figure and interval summaries

This section creates the SSRI overlap-days histogram and 4-day interval summary used for interpretation.


In [ ]:
# ============================================================
# FIGURE - Overlap days distribution for SSRI to single LOT2
# Recent 5-year index cohort
# Statistics and table use overlap_days > 0 only
# Plot range restricted to 1-730 days
# ============================================================

import matplotlib.pyplot as plt
import numpy as np


# ------------------------------------------------------------
# 1. Use the same SSRI single-LOT2 data as Table 7A
# ------------------------------------------------------------

ssri_lot2_plot_order = [
    "Atypical Antidepressants",
    "SNRI",
    "TCA"
]


ssri_single_lot2_overlap_gt0 = (
    table7_ssri_single_lot2[
        (
            table7_ssri_single_lot2["overlap_days"] > 0
        )
        & (
            table7_ssri_single_lot2["lot2_class"].isin(
                ssri_lot2_plot_order
            )
        )
    ]
    .copy()
)


# ------------------------------------------------------------
# 2. Keep outliers in the table/summary,
#    restrict plot range only
# ------------------------------------------------------------

ssri_single_lot2_overlap_gt0_plot = (
    ssri_single_lot2_overlap_gt0[
        ssri_single_lot2_overlap_gt0["overlap_days"]
        .between(
            1,
            730
        )
    ]
    .copy()
)


print(
    "SSRI single LOT2 patients with overlap_days > 0:",
    ssri_single_lot2_overlap_gt0["person_id"].nunique()
)

print(
    "Patients included in plot range 1-730 days:",
    ssri_single_lot2_overlap_gt0_plot["person_id"].nunique()
)

print(
    "Patients with overlap_days > 730 excluded from plot only:",
    ssri_single_lot2_overlap_gt0["person_id"].nunique()
    - ssri_single_lot2_overlap_gt0_plot["person_id"].nunique()
)


# Optional check by LOT2 class
plot_count_check = (
    ssri_single_lot2_overlap_gt0
    .groupby(
        "lot2_class",
        as_index=False
    )
    .agg(
        overlap_gt0_n=("person_id", "nunique"),
        min_overlap_days=("overlap_days", "min"),
        max_overlap_days=("overlap_days", "max")
    )
)

display(plot_count_check)


# ------------------------------------------------------------
# 3. Plot histogram panels
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    len(ssri_lot2_plot_order),
    figsize=(18, 5),
    sharex=True
)


# 3-day bins preserve the early-overlap pattern better than 7-day bins,
# while avoiding the overly thin bars from 1-day bins.
bins = np.arange(
    1,
    732,
    3
)


x_ticks = [
    7,
    14,
    30,
    60,
    90,
    180,
    365,
    730
]


for ax, lot2_class in zip(
    axes,
    ssri_lot2_plot_order
):

    plot_data = (
        ssri_single_lot2_overlap_gt0_plot[
            ssri_single_lot2_overlap_gt0_plot["lot2_class"]
            == lot2_class
        ]["overlap_days"]
    )

    ax.hist(
        plot_data,
        bins=bins,
        edgecolor="white",
        linewidth=0.2,
        rwidth=0.9
    )

    ax.set_title(
        lot2_class,
        fontsize=11
    )

    ax.set_xlim(
        1,
        730
    )

    ax.set_xticks(
        x_ticks
    )

    ax.set_xlabel(
        "Overlap days"
    )

    ax.grid(
        axis="y",
        alpha=0.3
    )


axes[0].set_ylabel(
    "Patient count"
)


fig.suptitle(
    "Overlap Days Distribution for SSRI to Single LOT2 Class: Recent 5-Year Index Cohort",
    fontsize=13,
    y=1.05
)


fig.text(
    0.5,
    0.96,
    (
        "Only patients with overlap_days > 0 are plotted. "
        "The x-axis is restricted to 1-730 days; patients with overlap_days > 730 days "
        "are excluded from the plot."
    ),
    ha="center",
    fontsize=10
)


plt.tight_layout()


# ------------------------------------------------------------
# 4. Export figure
# ------------------------------------------------------------

figure_path = os.path.join(
    output_dir,
    "Table7_SSRI_single_LOT2_overlap_days_distribution_recent5y.png"
)


plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)


plt.show()


print(
    "Figure exported:",
    figure_path
)

In [ ]:
# ============================================================
# SUMMARY - SSRI single LOT2 overlap_days > 0
# Count patients by LOT2 class and 4-day overlap interval
# ============================================================

import pandas as pd
import numpy as np
import os


# ------------------------------------------------------------
# 1. Use the same data as the figure/table
# ------------------------------------------------------------

ssri_lot2_summary_order = [
    "Atypical Antidepressants",
    "SNRI",
    "TCA"
]


ssri_single_lot2_overlap_gt0 = (
    table7_ssri_single_lot2[
        (
            table7_ssri_single_lot2["overlap_days"] > 0
        )
        & (
            table7_ssri_single_lot2["lot2_class"].isin(
                ssri_lot2_summary_order
            )
        )
    ]
    .copy()
)


ssri_single_lot2_overlap_gt0["overlap_days"] = pd.to_numeric(
    ssri_single_lot2_overlap_gt0["overlap_days"],
    errors="coerce"
).astype(int)


print(
    "SSRI single LOT2 patients with overlap_days > 0:",
    ssri_single_lot2_overlap_gt0["person_id"].nunique()
)


# ------------------------------------------------------------
# 2. Create 4-day bins for 1-730 days
#    and keep >730 as a separate outlier bin
# ------------------------------------------------------------

bin_width = 4


plot_max_day = 730


# Example bins:
# 1-4, 5-8, 9-12, ..., 729-730
bin_starts = list(
    range(
        1,
        plot_max_day + 1,
        bin_width
    )
)


bin_rows = []

for start_day in bin_starts:

    end_day = min(
        start_day + bin_width - 1,
        plot_max_day
    )

    bin_rows.append({
        "bin_start": start_day,
        "bin_end": end_day,
        "overlap_day_interval": f"{start_day}-{end_day}"
    })


bin_lookup = pd.DataFrame(
    bin_rows
)


def assign_overlap_interval(days):
    """
    Assign overlap_days to a 4-day interval.
    """

    if pd.isna(days):
        return np.nan

    if days > plot_max_day:
        return ">730"

    bin_start = (
        ((int(days) - 1) // bin_width)
        * bin_width
        + 1
    )

    bin_end = min(
        bin_start + bin_width - 1,
        plot_max_day
    )

    return f"{bin_start}-{bin_end}"


ssri_single_lot2_overlap_gt0["overlap_day_interval"] = (
    ssri_single_lot2_overlap_gt0["overlap_days"]
    .apply(assign_overlap_interval)
)


# ------------------------------------------------------------
# 3. Count patients by LOT2 class and interval
# ------------------------------------------------------------

overlap_interval_summary = (
    ssri_single_lot2_overlap_gt0
    .groupby(
        [
            "lot2_class",
            "overlap_day_interval"
        ],
        as_index=False
    )
    .agg(
        patient_n=("person_id", "nunique")
    )
)


# ------------------------------------------------------------
# 4. Create complete grid so missing intervals show as 0
# ------------------------------------------------------------

interval_order = (
    bin_lookup["overlap_day_interval"].tolist()
    + [">730"]
)


complete_grid = pd.MultiIndex.from_product(
    [
        ssri_lot2_summary_order,
        interval_order
    ],
    names=[
        "lot2_class",
        "overlap_day_interval"
    ]
).to_frame(index=False)


overlap_interval_summary = (
    complete_grid
    .merge(
        overlap_interval_summary,
        on=[
            "lot2_class",
            "overlap_day_interval"
        ],
        how="left"
    )
)


overlap_interval_summary["patient_n"] = (
    overlap_interval_summary["patient_n"]
    .fillna(0)
    .astype(int)
)


# ------------------------------------------------------------
# 5. Add percent within each LOT2 class
# ------------------------------------------------------------

lot2_total_n = (
    ssri_single_lot2_overlap_gt0
    .groupby(
        "lot2_class",
        as_index=False
    )
    .agg(
        lot2_overlap_gt0_n=("person_id", "nunique")
    )
)


overlap_interval_summary = (
    overlap_interval_summary
    .merge(
        lot2_total_n,
        on="lot2_class",
        how="left"
    )
)


overlap_interval_summary["pct_within_lot2"] = (
    overlap_interval_summary["patient_n"]
    / overlap_interval_summary["lot2_overlap_gt0_n"]
    * 100
)


overlap_interval_summary["pct_within_lot2"] = (
    overlap_interval_summary["pct_within_lot2"]
    .round(2)
)


overlap_interval_summary["patient_n_pct_display"] = (
    overlap_interval_summary["patient_n"].astype(str)
    + " ("
    + overlap_interval_summary["pct_within_lot2"].astype(str)
    + "%)"
)


# ------------------------------------------------------------
# 6. Add sorting
# ------------------------------------------------------------

lot2_order_df = pd.DataFrame({
    "lot2_class": ssri_lot2_summary_order,
    "lot2_order": range(len(ssri_lot2_summary_order))
})


interval_order_df = pd.DataFrame({
    "overlap_day_interval": interval_order,
    "interval_order": range(len(interval_order))
})


overlap_interval_summary = (
    overlap_interval_summary
    .merge(
        lot2_order_df,
        on="lot2_class",
        how="left"
    )
    .merge(
        interval_order_df,
        on="overlap_day_interval",
        how="left"
    )
    .sort_values(
        [
            "lot2_order",
            "interval_order"
        ]
    )
    .drop(
        columns=[
            "lot2_order",
            "interval_order"
        ]
    )
    .reset_index(drop=True)
)


with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None
):
    display(overlap_interval_summary)

In [ ]:
# ============================================================
# WIDE FORMAT - easier to inspect or paste into Excel
# ============================================================

overlap_interval_summary_wide = (
    overlap_interval_summary
    .pivot(
        index="overlap_day_interval",
        columns="lot2_class",
        values="patient_n"
    )
    .reset_index()
)


# Force interval order
overlap_interval_summary_wide = (
    interval_order_df
    .merge(
        overlap_interval_summary_wide,
        on="overlap_day_interval",
        how="left"
    )
    .sort_values("interval_order")
    .drop(columns="interval_order")
    .reset_index(drop=True)
)


overlap_interval_summary_wide[
    ssri_lot2_summary_order
] = (
    overlap_interval_summary_wide[
        ssri_lot2_summary_order
    ]
    .fillna(0)
    .astype(int)
)


with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None
):
    display(overlap_interval_summary_wide)

In [ ]:
# ============================================================
# FIGURE - Overlap days distribution for SSRI to single LOT2
# Recent 5-year index cohort
# 4-day bins with Q1 / median / Q3 reference lines
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import os


# ------------------------------------------------------------
# 1. Define LOT2 classes to plot
# ------------------------------------------------------------

ssri_lot2_plot_order = [
    "Atypical Antidepressants",
    "SNRI",
    "TCA"
]


# ------------------------------------------------------------
# 2. Use the same data source as Table 7A
#    SSRI LOT1, single LOT2, overlap_days > 0 only
# ------------------------------------------------------------

ssri_single_lot2_overlap_gt0 = (
    table7_ssri_single_lot2[
        (
            table7_ssri_single_lot2["overlap_days"] > 0
        )
        & (
            table7_ssri_single_lot2["lot2_class"].isin(
                ssri_lot2_plot_order
            )
        )
    ]
    .copy()
)


ssri_single_lot2_overlap_gt0["overlap_days"] = pd.to_numeric(
    ssri_single_lot2_overlap_gt0["overlap_days"],
    errors="coerce"
).astype(int)


# ------------------------------------------------------------
# 3. Restrict plot range only
#    Patients >730 days are retained in Table 7A summary
#    but excluded from this figure
# ------------------------------------------------------------

ssri_single_lot2_overlap_gt0_plot = (
    ssri_single_lot2_overlap_gt0[
        ssri_single_lot2_overlap_gt0["overlap_days"]
        .between(
            1,
            730
        )
    ]
    .copy()
)


print(
    "SSRI single LOT2 patients with overlap_days > 0:",
    ssri_single_lot2_overlap_gt0["person_id"].nunique()
)

print(
    "Patients included in plot range 1-730 days:",
    ssri_single_lot2_overlap_gt0_plot["person_id"].nunique()
)

print(
    "Patients with overlap_days > 730 excluded from plot only:",
    ssri_single_lot2_overlap_gt0["person_id"].nunique()
    - ssri_single_lot2_overlap_gt0_plot["person_id"].nunique()
)


# ------------------------------------------------------------
# 4. Count check by LOT2 class
# ------------------------------------------------------------

plot_count_check = (
    ssri_single_lot2_overlap_gt0
    .groupby(
        "lot2_class",
        as_index=False
    )
    .agg(
        overlap_gt0_n=("person_id", "nunique"),
        min_overlap_days=("overlap_days", "min"),
        max_overlap_days=("overlap_days", "max")
    )
)


plot_count_check["lot2_order"] = (
    plot_count_check["lot2_class"]
    .map(
        {
            class_name: i
            for i, class_name in enumerate(ssri_lot2_plot_order)
        }
    )
)


plot_count_check = (
    plot_count_check
    .sort_values("lot2_order")
    .drop(columns="lot2_order")
    .reset_index(drop=True)
)


display(plot_count_check)


# ------------------------------------------------------------
# 5. Summary values for Q1 / median / Q3 lines
#    These should come from Table 7A summary, where stats are
#    calculated among overlap_days > 0 only.
# ------------------------------------------------------------

summary_for_lines = (
    table7_ssri_single_lot2_summary[
        [
            "lot2_class",
            "q1_overlap_days",
            "median_overlap_days",
            "q3_overlap_days"
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# 6. Create histogram panels
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    len(ssri_lot2_plot_order),
    figsize=(18, 5),
    sharex=True
)


# 4-day bins:
# [1, 5)   = 1, 2, 3, 4
# [5, 9)   = 5, 6, 7, 8
# [9, 13)  = 9, 10, 11, 12
# ...
# Because data are restricted to <=730, the final plotted range
# still ends at 730.
bins = np.arange(
    1,
    735,
    4
)


x_ticks = [
    7,
    14,
    30,
    60,
    90,
    180,
    365,
    730
]


bar_color = "#4C78A8"
q_line_color = "#2E7D32"
median_line_color = bar_color


for ax, lot2_class in zip(
    axes,
    ssri_lot2_plot_order
):

    plot_data = (
        ssri_single_lot2_overlap_gt0_plot[
            ssri_single_lot2_overlap_gt0_plot["lot2_class"]
            == lot2_class
        ]["overlap_days"]
    )

    # Histogram
    ax.hist(
        plot_data,
        bins=bins,
        color=bar_color,
        edgecolor="white",
        linewidth=0.2,
        rwidth=0.9
    )

    # Get Q1, median, Q3
    line_row = summary_for_lines[
        summary_for_lines["lot2_class"] == lot2_class
    ].iloc[0]

    q1_value = line_row["q1_overlap_days"]
    median_value = line_row["median_overlap_days"]
    q3_value = line_row["q3_overlap_days"]

    # Q1 line: green dotted
    ax.axvline(
        q1_value,
        color=q_line_color,
        linestyle=":",
        linewidth=1.2,
        label="Q1"
    )

    # Median line: same color as bars, slightly thicker dashed
    ax.axvline(
        median_value,
        color=median_line_color,
        linestyle="--",
        linewidth=1.6,
        label="Median"
    )

    # Q3 line: green dotted
    ax.axvline(
        q3_value,
        color=q_line_color,
        linestyle=":",
        linewidth=1.2,
        label="Q3"
    )

    # Text box
    ax.text(
        0.98,
        0.92,
        (
            f"Median [Q1, Q3]\n"
            f"{median_value:.1f} "
            f"[{q1_value:.1f}, {q3_value:.1f}]"
        ),
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=9,
        bbox=dict(
            facecolor="white",
            edgecolor="gray",
            alpha=0.8
        )
    )

    ax.set_title(
        lot2_class,
        fontsize=11
    )

    ax.set_xlim(
        1,
        730
    )

    ax.set_xticks(
        x_ticks
    )

    ax.set_xlabel(
        "Overlap days"
    )

    ax.grid(
        axis="y",
        alpha=0.3
    )


axes[0].set_ylabel(
    "Patient count"
)


# ------------------------------------------------------------
# 7. Shared legend
# ------------------------------------------------------------

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.90),
    ncol=3,
    frameon=False
)


# ------------------------------------------------------------
# 8. Title and footnote
# ------------------------------------------------------------

fig.suptitle(
    "Overlap Days Distribution for SSRI to Single LOT2 Class: Recent 5-Year Index Cohort",
    fontsize=13,
    y=1.05
)


fig.text(
    0.5,
    0.96,
    (
        "Only patients with overlap_days > 0 are plotted. "
        "Blue dashed line = median; green dotted lines = Q1 and Q3. "
        "The x-axis is restricted to 1-730 days; patients with overlap_days > 730 days are excluded from the plot only."
    ),
    ha="center",
    fontsize=10
)


plt.tight_layout(
    rect=[
        0,
        0,
        1,
        0.88
    ]
)


# ------------------------------------------------------------
# 9. Export figure
# ------------------------------------------------------------

figure_path = os.path.join(
    output_dir,
    "Table7_SSRI_single_LOT2_overlap_days_distribution_recent5y_4day_bins_Q1_median_Q3.png"
)


plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)


plt.show()


print(
    "Figure exported:",
    figure_path
)